In [2]:
import os, subprocess
from astropy.io import fits
from datetime import datetime, timedelta
from sunpy.coordinates.sun import carrington_rotation_time
import astropy.units as u
from astropy.time import Time, TimeDelta, TimeDatetime
import sunpy.map
import numpy as np
from scipy import interpolate

In [3]:
# set_info -c ds="mps_loeschl.phi_m720s_test" T_REC="2021.02.19_12:30:03_TAI" magnetogram=solo_L2_phi-fdt-blos_20210219T123003_V202107130922C_0142190403_drms.fits

In [4]:
# jv2ts in=mps_loeschl.phi_m720s_test['2021.02.19_12:30:03_TAI'] v2hout=mps_loeschl.Ml_hiresmap_720s_test histlink=none TSTART='2021.02.19_12:30:03_TAI' TTOTAL='12m' TCHUNK='12m' MAPMMAX=5402 SINBDIVS=2160 LGSHIFT=3 CARRSTRETCH=1 MCORLEV=1 MAPRMAX=0.998 MAPLGMAX=90.0 MAPLGMIN=-90 MAPBMAX=90.0 VCORLEV=0 NAN_BEYOND_RMAX=1 FORCEOUTPUT=1

In [5]:
# resizemappingmag in=mps_loeschl.Ml_hiresmap_720s_test['2021.02.19_12:30:03_TAI'] out=mps_loeschl.Ml_remap_720s_test nbin=3 

In [6]:
# show_info -iP mps_loeschl.Ml_hiresmap_720s_test['2021.02.19_12:30:03_TAI']

In [7]:
# - Header can contain additional keywords that are not required for the data series
# - DRMS modules don't crash if they try to copy non existing keywords!

In [8]:
# TEMPORARY FIXES
# TODO FIX FILENAME ONCE WE HAVE GOOD L2 HEADERS
# TODO TEMPORARY: Fix CAR_ROT bug
# TODO Extract outdated filename / Find updated source file from observation date/time of outdated source file

# DRMS Compatible FITS

In [34]:
def calc_trec(crln_obs, car_rot, verbose=False):
    # helper function to prepare hmi data for interp_phi2hmi() interpolation

    # remap and >180 means that HMI_PAST is the previous CAR_ROT and HMI_FUTR is the current CAR_ROT
    # remap and <180 means that HMI_FUTR is the current CAR_ROT and HMI_PAST is the previous CAR_ROT
    car_rot = 2258
    
    # THIS IS LOGIC DOESN'T MAKE SENSE FOR THE BOTTOM LEFT QUARTER OF OBSERVATIONS (ORBIT_PLOTS)
    if False:# crln_obs > 180:
        # t defined by when HMI sees it (future/past)
        #t0 = carrington_rotation_time(car_rot-1) # past / current
        #t1 = carrington_rotation_time(car_rot)   # future
        
        t0 = carrington_rotation_time(car_rot-1) # past / current CAR_ROT
        t1 = carrington_rotation_time(car_rot)   # future
        t2 = carrington_rotation_time(car_rot+1) # future end point
        
        trec_hmi = interp_phi2hmi(crln_obs, t0, t1, verbose)
        hmi_next = interp_phi2hmi(crln_obs, t1, t2)
        hmi_prev = trec_hmi
        car_rot -= 1
        
    else:
        t0 = carrington_rotation_time(car_rot-1) # past  
        t1 = carrington_rotation_time(car_rot)   # future / current
        t2 = carrington_rotation_time(car_rot+1) # future end point

        trec_hmi = interp_phi2hmi(crln_obs, t1, t2, verbose)
        hmi_prev = interp_phi2hmi(crln_obs, t0, t1)
        hmi_next = trec_hmi
        

    print(trec_hmi, hmi_prev, hmi_next, crln_obs, car_rot)
    return trec_hmi, hmi_prev, hmi_next, car_rot

"""
def calc_trec_rev(crln_obs, car_rot, verbose=False):
    # helper function to prepare hmi data for interp_phi2hmi() interpolation

    # remap and >180 means that HMI_PAST is the previous CAR_ROT and HMI_FUTR is the current CAR_ROT
    # remap and <180 means that HMI_FUTR is the current CAR_ROT and HMI_PAST is the previous CAR_ROT
    
    t0 = carrington_rotation_time(car_rot-1) # past / current CAR_ROT
    t1 = carrington_rotation_time(car_rot)   # future
    t2 = carrington_rotation_time(car_rot+1) # future end point
    
    # THIS IS LOGIC DOESN'T MAKE SENSE FOR THE BOTTOM LEFT QUARTER OF OBSERVATIONS (ORBIT_PLOTS)
    if crln_obs > 180:
        # t defined by when HMI sees it (future/past)
        #t0 = carrington_rotation_time(car_rot-1) # past / current
        #t1 = carrington_rotation_time(car_rot)   # future
    
        trec_hmi = interp_phi2hmi(crln_obs, t0, t1, verbose)
        hmi_next = interp_phi2hmi(crln_obs, t1, t2)
        hmi_prev = trec_hmi
        car_rot -= 1
        
    else:
        
        trec_hmi = interp_phi2hmi(crln_obs, t1, t2, verbose)
        hmi_prev = interp_phi2hmi(crln_obs, t0, t1)
        hmi_next = trec_hmi
        
    print(trec_hmi, hmi_prev, hmi_next, crln_obs, car_rot)
    return trec_hmi, hmi_prev, hmi_next, car_rot
"""

def interp_phi2hmi(crln_obs, t0, t1, verbose=False):
    # Interpolate T_REC of PHI CRLN_OBS onto HMI CRLN_OBS
    dt_hmi   = np.array([])
    trec_hmi = np.array([])
    crln_hmi = np.array([])

    hmi_times = "%s-%s" %(t0.datetime.strftime("%Y.%m.%d_%H:%M:%S_TAI"), t1.datetime.strftime("%Y.%m.%d_%H:%M:%S_TAI"))
    hmi_data, n = get_drms_keywords(hmi_times, "hmi.m_720s") #"mps_loeschl.Ml_remap_720s")
    if verbose: print('Mapping PHI to CR %s in HMI period %s...' %(car_rot, hmi_times))
          
    for line in hmi_data:
        # CRLN_OBS will be NaN if no observation is available for a timeslot -> nan filter required
        if np.isnan(float(line['CRLN_OBS'])): continue 
        
        trec_hmi = np.append(trec_hmi, datetime.strptime(line['T_REC'], "%Y.%m.%d_%H:%M:%S_TAI"))
        dt_hmi   = np.append(dt_hmi, ((trec_hmi[-1] - t0.datetime).days +(trec_hmi[-1] - t0.datetime).seconds/(3600*24)))    
        crln_hmi = np.append(crln_hmi, float(line['CRLN_OBS']))

    # Interpolation fails if the HMI onto which I want to map is not complete yet!
    # this will happen whenever we try to preview ongoing carrington rotations
    # the timeslot interpolation must be based on an extrapolation for the remaining HMI time slots/clrn obs
    
    # extrapolation if trec_hmi[-1]-trec_hmi[0] < 1 month
    crd = t1-t0 # carrington rotation duration
    
    hmi_end = datetime.strptime(hmi_data[-1]['T_REC'], "%Y.%m.%d_%H:%M:%S_TAI")
    dt = (hmi_end-t0.datetime).days +(hmi_end-t0.datetime).seconds/(3600*24)
    tstep = timedelta(minutes=12)
    
    # Extrapolation of T_REC/CRLN_OBS in 12 minute steps
    if dt < crd:
        crln_fit = interpolate.interp1d(dt_hmi, crln_hmi, fill_value = "extrapolate")
        nsteps = np.ceil(((crd-dt)*(24*3600)).value/720).astype(int) # difference in seconds
        
        for i in range(1, nsteps):
            #hmi_data.append({'T_REC':(hmi_end+i*tstep).strftime("%Y.%m.%d_%H:%M:%S_TAI"), 'CRLN_OBS':crln_fit[i-1], 'CAR_ROT':hmi_data[0]['CAR_ROT']})
            dt_hmi   = np.append(dt_hmi, (((hmi_end+i*tstep) - t0.datetime).days +((hmi_end+i*tstep) - t0.datetime).seconds/(3600*24)))    
            trec_hmi = np.append(trec_hmi, (hmi_end+i*tstep).strftime("%Y.%m.%d_%H:%M:%S_TAI"))
            
            crln =  crln_fit(dt_hmi[-1])
            if crln < 0: crln += 360
            crln_hmi = np.append(crln_hmi, crln)
    
    t_interp = timedelta(days=np.interp(crln_obs, crln_hmi, dt_hmi, period=360))
    trec_phi = t0.datetime+t_interp
    trec_hmi = trec_phi.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    
    return trec_hmi
    
def get_drms_keywords(inRecs, input_ds):

    #inRecs = "2014.05.12_12:00:00_TAI, 2014.05.13_00:00:00_TAI, 2014.05.13_12:00:00_TAI, 2014.05.14_00:00:00_TAI" # input argument
    show_info = 'show_info %s["%s"] key="T_REC,CRLN_OBS,CAR_ROT"'
    
    #-P for path and -A for segment
    si_out = subprocess.check_output(show_info %(input_ds, inRecs) , shell=True)[:-1].decode("utf-8")
    raw = si_out.split('\n')

    formatted = [] 
    drms_param = []
    
    nRecs = 0
    keys = raw[0].split('\t')
    
    for line in raw[1:]:  
        formatted = line.split('\t') # [CALVER64, T_REC, QUALITY, FDRADIAL, CARSTRCH, DIFROT_A, DIFROT_B, DIFROT_C, CRVAL1, CRLN_OBS, CAR_ROT, MAPLGMAX, MAPLGMIN, I_DREC]

        dict_tmp = {}

        for i, key in enumerate(keys):
            
            if key == "magnetogram" or key == 'Ml':
                key = "PATH"
                
            if formatted[i].strip() == "InvalidKeyname":
                dict_tmp[key] = 0
            else:
                dict_tmp[key] = formatted[i]
   
        drms_param.append(dict_tmp)
        nRecs += 1

    return drms_param, nRecs


In [44]:
#path = "../output/data/phi/feb2021_rev02/"
#path = "../output/data/phi/feb2021_trl_v01/"
#path =  "../output/data/phi/FDT_test_release_sup_conj_2021/"
path =  "../output/data/phi/FDT_test_release_june_2022_defringed/"
dbpath = "/data/solo/phi/data/fmdb/l1/%s/%s"
#dbpath = "/www/docs/data/outgoing/valori/PHI_LL/2022-02-27/"

old_header = False

files = os.listdir(path)
fitsfiles = [file for file in files if file.endswith(".fits") or file.endswith(".fits.gz")]

if fitsfiles[0].endswith(".fits"):
    n_end = 5
else:
    n_end = 8

trecs = []
clons = []

for file in fitsfiles:
    l2 = fits.open(path+file)
  
    print("Processing %s ..." %file)
    
    # David's L2 files were processed with an old header version. 
    # Find respective UPDATED L1 files and copy the missing keywords
    # L1 parent FILENAME is outdated. Find source file with updated processing
    
    # solo_L1_phi-fdt-ilam_obsdateTobstime_processingdate_noclue.fits.gz
    # solo_L1_phi-fdt-ilam_20210205T200002_V202108301608C_0142050411.fits.gz
    # TODO
    # Extract outdated filename
    if old_header:
        date = l2[0].header['FILENAME'][21:29] # eg '20210219'
        date_path = "%s-%s-%s" %(date[:4], date[4:6], date[6:8])

        # Find updated source file from observation date/time of outdated source file
        tmpfiles = os.listdir(dbpath %(date_path, ""))
        for tmpfile in tmpfiles:
            if l2[0].header['FILENAME'][:37] in tmpfile: # FILENAME[:37] eg: 'solo_L1_phi-fdt-ilam_20210205T200002_'
                break
    
    prim = fits.PrimaryHDU()
    l2drms = fits.CompImageHDU(data=l2[0].data.astype(np.int32))

    # open updated source file
    # TODO CR2240
    if old_header: l1 = fits.open(dbpath %(date_path, tmpfile))

    l2drms.header.append(('', '', ''), end=True)
    l2drms.header.append(('', '  / HMI Compatibility', ''), end=True)
    
    # TODO TEMPORARY: Fix CAR_ROT bug
    if l2[0].header['CAR_ROT'] == 2239: 
        l2[0].header['CAR_ROT'] = 2240
    
    if old_header:
        if l2[0].header['CRLN_OBS'] == 358.81235 and l2[0].header['CAR_ROT'] == 2240:
            l2[0].header['CAR_ROT'] = 2241
            
    else:
        if l2[0].header['CRLN_OBS'] == 358.80689 and l2[0].header['CAR_ROT'] == 2240:
            l2[0].header['CAR_ROT'] = 2241
      
    # T_REC / T_OBS
    #DATE-AVG= '2021-02-05T20:00:45.616' / [UTC] Average time of observation     
    #T_OBS   = '2021.02.28_07:11:55.437_TAI' / [TAI] nominal time 
    #T_REC   = '2021.02.28_07:12:00.000_TAI' / [TAI] Slot time    
    # Conversion for date format / UTC to TAI / round to the next 12 minute slot for T_REC
    #trec = datetime.strptime(l2[0].header['DATE-AVG'],   "%Y-%m-%dT%H:%M:%S.%f")
    tobs = datetime.strptime(l2[0].header['DATE-AVG'],   "%Y-%m-%dT%H:%M:%S.%f")
    
    #utc2tai = timedelta(0, 37)                 # use for utc2tai conversion
    #trec = trec + utc2tai                      # use for utc2tai conversion
    #trec = trec.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    tobs = tobs.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    
    # T_REC Interpolation       
    trec, hmi_prev, hmi_next, car_rot = calc_trec(float(l2[0].header['CRLN_OBS']), int(l2[0].header['CAR_ROT']), verbose=False)
    print(trec, car_rot, l2[0].header['CRLN_OBS'])
    trecs.append(trec)
    clons.append(l2[0].header['CRLN_OBS'])
    
    l2drms.header.append(('T_REC', trec, ''), end=True)
    l2drms.header.append(('T_OBS', tobs, ''), end=True)  # required for JV2TS
    
    l2drms.header.append(('TRECEPOC', '1993.01.01_00:00:00_TAI', 'Time of origin'), end=True)
    l2drms.header.append(('TRECSTEP', 720.0,  'ts_eq step'), end=True)
    
    # these two keywords are redundant with T_REC and T_OBS
    l2drms.header.append(('HMI_PREV', hmi_prev, 'Previous HMI T_REC for this CRLN_OBS'), end=True) # HMI interpolated T_REC
    l2drms.header.append(('HMI_NEXT', hmi_next, 'Next HMI T_REC for this CRLN_OBS'), end=True) # real PHI observation date as backup
    
    # DATE
    l2drms.header.append(('DATE', l2[0].header['DATE'], "Date and time of FITS file creation, in UTC, in ISO-8601 format 'yyyy-mm-ddThh:mm:ss.sss'"), end=True)
    
    # DATE-OBS
    # DATE-BEG= '2021-02-05T20:00:02.906' / [UTC] Start time of observation 
    # DATE-OBS= '2021-02-28T07:10:33.400' / [ISO] Observation date {DATE__OBS}   
    # Conversion from BEG to OBS. Conversion from UTC to ISO
    date_obs = datetime.strptime(l2[0].header['DATE-BEG'],   "%Y-%m-%dT%H:%M:%S.%f")
    date_obs = date_obs.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    l2drms.header.append(('DATE-OBS', date_obs, 'DATE-OBS = DATE-AVG - EXPTIME/2.0'), end=True)

    # CADENCE
    # Missing, directly copy from HMI as a dummy value or extract daily? cadence from filenames
    l2drms.header.append(('CADENCE', 720.0, '[seconds] Observation cadence - DUMMY VALUE'), end=True)
    
    # TELESCOP
    l2drms.header.append(('TELESCOP', l2[0].header['TELESCOP'], 'SOLO/PHI/FDT, SOLO/PHI/HRT, HMI: SDO/HMI'), end=True)
    
    # INSTRUME
    l2drms.header.append(('INSTRUME', l2[0].header['INSTRUME'], 'PHI, HMI_SIDE1, HMI_FRONT2, HMI_COMBINED'), end=True)
    
    # WAVELNTH
    l2drms.header.append(('WAVELNTH', 6173.341, 'For PHI/HMI: 6173.3 Angstroms'), end=True)
    
    # QUALITY
    # Missing. add as dummy value = 0. Quality index of data used to genrate fd.B_720s (0x00000000) is good quality, (any nonzero value) should be investiaged in data documentation
    l2drms.header.append(('QUALITY', 0, 'Level 1.5 Quality - DUMMY VALUE'), end=True)

    # BUNIT
    l2drms.header.append(('BUNIT', l2[0].header['BUNIT'], 'BUNIT: physical units of each data segment'), end=True)
    
    # HISTORY
    # TODO FIX FILENAME ONCE WE HAVE GOOD L2 HEADERS
    # TODO include for CR2240
    if old_header:
        l2drms.header.append(('HISTORY', 'DRMS compatible FITS created from %s' %l1[0].header['FILENAME'], 'History of data'), end=True)
    else:
        l2drms.header.append(('HISTORY', 'DRMS compatible FITS created from %s' %l2[0].header['FILENAME'], 'History of data'), end=True)
    
    
    # COMMENT
    #l2drms.header.append(('COMMENT', l2[0].header['COMMENT'], 'Commentary on the data'), end=True)
    
    # CTYPE1
    l2drms.header.append(('CTYPE1', l2[0].header['CTYPE1'], 'CTYPE1: HPLN-TAN (SOLARX)'), end=True)
    
    # CTYPE2
    l2drms.header.append(('CTYPE2', l2[0].header['CTYPE2'], 'CTYPE2: HPLN-TAN (SOLARY)'), end=True)
    
    #CRPIX1
    l2drms.header.append(('CRPIX1', l2[0].header['CRPIX1'], 'CRPIX1: location of the Sun center in CCD x direction'), end=True)
    
    #CRPIX2
    l2drms.header.append(('CRPIX2', l2[0].header['CRPIX2'], 'CRPIX2: location of the Sun center in CCD y direction'), end=True)
    
    #CRVAL1
    l2drms.header.append(('CRVAL1', l2[0].header['CRVAL1'], 'CRVAL1: x origin - center of the solar disk'), end=True)
    
    #CRVAL2
    l2drms.header.append(('CRVAL2', l2[0].header['CRVAL2'], 'CRVAL2: y origin - center of the solar disk'), end=True)
    
    #CDELT1
    l2drms.header.append(('CDELT1', l2[0].header['CDELT1'], 'Image scale in the x direction'), end=True)
    
    #CDELT2
    l2drms.header.append(('CDELT2', l2[0].header['CDELT2'], 'Image scale in the y direction'), end=True)
    
    #CUNIT1
    l2drms.header.append(('CUNIT1', l2[0].header['CUNIT1'], 'CUNIT1: arcsec'), end=True)
    
    #CUNIT2
    l2drms.header.append(('CUNIT2', l2[0].header['CUNIT2'], 'CUNIT2: arcsec'), end=True)
    
    # CROTA 
    # CROTA2
    # CROTA was renamed to CROTA2 in Dietmar's current L2 header. Rename here for now. CHECK IF THE ROTATION MAKES SENSE AFTER THE PROJECTION by comparing the projections of equal CRLN_OBS in PHI and HMI
    l2drms.header.append(('CROTA2', l2[0].header['CROTA'], '[deg] Rotation angle'), end=True)

    #CRDER1
    #l2drms.header.append(('CRDER1', l1[0].header['CRDER1'], 'CRDER1: estimate of random error in coordinate x'), end=True)
    
    #CRDER2
    #l2drms.header.append(('CRDER2', l1[0].header['CRDER2'], 'CRDER2: estimate of random error in coordinate y'), end=True)
    
    #CSYSER1
    #l2drms.header.append(('CSYSER1', l2[0].header['CSYSER1'], 'CSYSER1: estimate of systematic error in coordinate x'), end=True)
    
    #CSYSER2
    #l2drms.header.append(('CSYSER2', l2[0].header['CSYSER2'], 'CSYSER2: estimate of systematic error in coordinate y'), end=True)
    
    #WCSNAME
    l2drms.header.append(('WCSNAME', l2[0].header['WCSNAME'], 'WCS system name'), end=True)
    
    #DSUN_OBS
    l2drms.header.append(('DSUN_OBS', l2[0].header['DSUN_OBS'], 'Distance from SDO to Sun center.'), end=True)
    
    #RSUN_REF
    l2drms.header.append(('RSUN_REF', l2[0].header['RSUN_REF'], 'Reference radius of the Sun: 696,000,000.0 m'), end=True)
    
    #CRLN_OBS
    l2drms.header.append(('CRLN_OBS', l2[0].header['CRLN_OBS'], 'Carrington longitude of HMI'), end=True)
    
    #CRLT_OBS
    l2drms.header.append(('CRLT_OBS', l2[0].header['CRLT_OBS'], 'Carrington latitude of HMI'), end=True)
    
    #CAR_ROT        
    l2drms.header.append(('CAR_ROT', l2[0].header['CAR_ROT'], 'Carrington rotation number of CRLN_OBS'), end=True)
    l2drms.header.append(('CAR_ROT2', car_rot, 'Carrington rotation number of synoptic map'), end=True)
    
    # OBS_VW
    # OBS_VR
    # OBS_VN
    # Both are missing in the OLD HEADER (David's) but should't in the future. Fetch keywords from L1 data, which is the one listed in the processed FILENAME history keyword
    # eg.: FILENAME solo_L1_phi-fdt-ilam_20210205T200002_V202108301608C_0142050411.fits
    
    # TODO include for CR2240
    if old_header:
        l2drms.header.append(('OBS_VR', l1[0].header['OBS_VR'], '[m/s] Radial velocity of S/C relative to Sun   '), end=True)
        l2drms.header.append(('OBS_VW', l1[0].header['OBS_VW'], '[m/s] Westward velocity of S/C relative to Sun '), end=True)
        l2drms.header.append(('OBS_VN', l1[0].header['OBS_VN'], '[m/s] Northward velocity of S/C relative to Sun'), end=True)
    else:
        l2drms.header.append(('OBS_VR', l2[0].header['OBS_VR'], '[m/s] Radial velocity of S/C relative to Sun   '), end=True)
        l2drms.header.append(('OBS_VW', l2[0].header['OBS_VW'], '[m/s] Westward velocity of S/C relative to Sun '), end=True)
        l2drms.header.append(('OBS_VN', l2[0].header['OBS_VN'], '[m/s] Northward velocity of S/C relative to Sun'), end=True)
    # RSUN_ARC
    # RSUN_OBS
    # Same keyword. Save RSUN_ARC as RSUN_OBS for HMI
    l2drms.header.append(('RSUN_OBS', l2[0].header['RSUN_ARC'], '[arcsec] angular radius of Sun.'), end=True)

    # DATAVALS
    # Actual number of data values in images (pixels)
    l2drms.header.append(('DATAVALS', l2[0].header['NAXIS1']*l2[0].header['NAXIS2'], 'Actual number of data values in images'), end=True)

    # MISSVALS
    # Missing values: TOTVALS - DATAVALS
    l2drms.header.append(('MISSVALS', 0, 'Missing values: TOTVALS - DATAVALS'), end=True)
    
    # DATAMIN
    l2drms.header.append(('DATAMIN', l2[0].header['DATAMIN'], 'Minimum value from pixels within 99% of solar radius'), end=True)
    
    # DATAMAX
    l2drms.header.append(('DATAMAX', l2[0].header['DATAMAX'], 'Maximum value from pixels within 99% of solar radius'), end=True)
    
    if not os.path.isdir(path+'drms/'):
        os.mkdir(path+'drms/')
    
    hdul = fits.HDUList([prim, l2drms])
    hdul.writeto(path+'drms/%s_drms.fits' %file[:-n_end], overwrite=True)
    #l1.close()
    
print('Done')

#ef make_fits(phi, hmi, dtype):
#   # phi ... image
#   # hmi ... image

#   prim = fits.PrimaryHDU(data=hmi[0].data, header=hmi[0].header)
#   cimg = fits.CompImageHDU(data=phi.astype(dtype), header=hmi[1].header[:-2]) # -2 skips old checksum and datasum
#   hdul = fits.HDUList([prim, cimg])
#       
#   return hdul

Processing solo_L2_phi-fdt-blos_20220603T031008_V202307311648_0246030501.fits.gz ...
2022.06.17_22:54:23_TAI 2022.05.21_18:04:12_TAI 2022.06.17_22:54:23_TAI 86.480925 2258
2022.06.17_22:54:23_TAI 2258 86.480925
Processing solo_L2_phi-fdt-blos_20220603T150009_V202307311648_0246030503.fits.gz ...
2022.06.18_10:56:25_TAI 2022.05.22_06:02:45_TAI 2022.06.18_10:56:25_TAI 79.86915 2258
2022.06.18_10:56:25_TAI 2258 79.86915
Processing solo_L2_phi-fdt-blos_20220604T030008_V202307311648_0246040501.fits.gz ...
2022.06.18_23:03:13_TAI 2022.05.22_18:13:49_TAI 2022.06.18_23:03:13_TAI 73.162503 2258
2022.06.18_23:03:13_TAI 2258 73.162503
Processing solo_L2_phi-fdt-blos_20220604T150009_V202307311648_0246040503.fits.gz ...
2022.06.19_11:15:52_TAI 2022.05.23_06:23:07_TAI 2022.06.19_11:15:52_TAI 66.453779 2258
2022.06.19_11:15:52_TAI 2258 66.453779
Processing solo_L2_phi-fdt-blos_20220605T030008_V202307311648_0246050501.fits.gz ...
2022.06.19_23:22:58_TAI 2022.05.23_18:34:16_TAI 2022.06.19_23:22:58_TAI 5

In [47]:
len(trecs), len(clons)

(52, 52)

In [48]:
trecs

['2022.06.17_22:54:23_TAI',
 '2022.06.18_10:56:25_TAI',
 '2022.06.18_23:03:13_TAI',
 '2022.06.19_11:15:52_TAI',
 '2022.06.19_23:22:58_TAI',
 '2022.06.20_11:36:08_TAI',
 '2022.06.20_23:53:39_TAI',
 '2022.06.21_04:17:58_TAI',
 '2022.06.22_00:04:49_TAI',
 '2022.06.22_12:18:51_TAI',
 '2022.06.22_18:23:04_TAI',
 '2022.06.23_00:26:52_TAI',
 '2022.06.23_06:33:54_TAI',
 '2022.06.23_12:41:16_TAI',
 '2022.06.23_18:45:28_TAI',
 '2022.06.24_00:49:36_TAI',
 '2022.06.24_06:56:58_TAI',
 '2022.05.28_08:15:44_TAI',
 '2022.05.28_14:22:55_TAI',
 '2022.05.28_20:26:46_TAI',
 '2022.05.29_02:31:46_TAI',
 '2022.05.29_08:40:07_TAI',
 '2022.05.29_14:47:11_TAI',
 '2022.05.29_20:51:05_TAI',
 '2022.05.30_02:56:29_TAI',
 '2022.05.30_09:05:03_TAI',
 '2022.05.30_15:12:00_TAI',
 '2022.05.30_21:15:56_TAI',
 '2022.05.31_03:21:44_TAI',
 '2022.05.31_09:30:30_TAI',
 '2022.06.01_09:56:24_TAI',
 '2022.06.01_16:03:02_TAI',
 '2022.06.01_22:07:09_TAI',
 '2022.06.02_04:13:44_TAI',
 '2022.06.02_10:22:46_TAI',
 '2022.06.02_16:29:1

In [49]:
clons

[86.480925,
 79.86915,
 73.162503,
 66.453779,
 59.743369,
 53.031033,
 46.223809,
 43.798889,
 32.884015,
 26.165025,
 22.804894,
 19.444456,
 16.083599,
 12.722313,
 9.3606778,
 5.9987202,
 2.6363213,
 359.27358,
 355.9105,
 352.54721,
 349.18343,
 345.81931,
 342.45488,
 339.09005,
 335.72493,
 332.35949,
 328.99372,
 325.62773,
 322.26125,
 318.89456,
 305.42488,
 302.05676,
 298.68835,
 295.31962,
 291.95063,
 288.58137,
 285.21185,
 281.84207,
 278.47203,
 275.10168,
 271.73122,
 268.36039,
 264.98932,
 261.61809,
 258.24661,
 254.87489,
 251.50294,
 248.13077,
 244.75838,
 241.38577,
 238.013,
 234.63993]

# DRMS Ingestion Scripts

In [38]:
#path = "../output/data/phi/feb2021_rev02/"
#path = "../output/data/phi/feb2021_trl_v01/"
#path =  "../output/data/phi/FDT_test_release_sup_conj_2021/"
path =  "../output/data/phi/FDT_test_release_june_2022_defringed/"

In [39]:
path_out = path+'drms/'
cr = 2258
rev = "_fdt_test_release_june_2022_defri"#"_rev00_r095"
Mr = True
maprmax = 0.9925 #0.998

files = os.listdir(path_out)
fitsfiles = [file for file in files if file.endswith(".fits")]
#mps_loeschl.Mr_hiresmap_CR2240_FDT_test_release_sup_conj_2021
if Mr:
    #data_series_m720s = "mps_loeschl.phi_m720s"
    #data_series_m720s = "mps_loeschl.phi_feb2021%s" %rev
    #data_series_jv2ts = "mps_loeschl.Mr_hiresmap_CR%s%s" %(cr, rev) #"mps_loeschl.Ml_hiresmap_720s_test"
    #data_series_remap = "mps_loeschl.Mr_remap_CR%s%s"    %(cr, rev) #"mps_loeschl.Ml_remap_720s_test"
    data_series_m720s = "mps_loeschl.phi_CR%s%s" %(cr, rev)
    data_series_jv2ts = "mps_loeschl.Mr_hiresmap_CR%s%s" %(cr, rev) #"mps_loeschl.Ml_hiresmap_720s_test"
    data_series_remap = "mps_loeschl.Mr_remap_CR%s%s"    %(cr, rev) #"mps_loeschl.Ml_remap_720s_test"
    proj = "Mr"
else:
    #data_series_m720s = "mps_loeschl.phi_feb2021%s" %rev
    #data_series_jv2ts = "mps_loeschl.Ml_hiresmap_CR%s%s"  %(cr, rev) #"mps_loeschl.Ml_hiresmap_720s_test"
    #data_series_remap = "mps_loeschl.Ml_remap_CR%s%s"     %(cr, rev) #"mps_loeschl.Ml_remap_720s_test"
    
    data_series_m720s = "mps_loeschl.phi_CR%s%s" %(cr, rev)
    data_series_jv2ts = "mps_loeschl.Ml_hiresmap_CR%s%s" %(cr, rev) #"mps_loeschl.Ml_hiresmap_720s_test"
    data_series_remap = "mps_loeschl.Ml_remap_CR%s%s"    %(cr, rev) #"mps_loeschl.Ml_remap_720s_test"
    proj = "Ml"

setinfo_out = open(path_out+'0_set_info.sh', 'w')
setinfo_out.write('#!/bin/bash\n')
#TODO segment names
set_info = 'set_info -c ds="%s" T_REC="%s" magnetogram=%s >> set_info.log 2>&1\n'

#TODO different combined remapping module
jv2ts_out = open(path_out+'1_jv2ts_%s.sh'%proj, 'w')
jv2ts_out.write('#!/bin/bash\n')

if Mr: jv2ts = "jv2ts in=%s['%s'] v2hout=%s histlink=none TSTART='%s' TTOTAL='12m' TCHUNK='12m' MAPMMAX=5402 SINBDIVS=2160 LGSHIFT=3 CARRSTRETCH=1 MCORLEV=2 MAPRMAX=%s MAPLGMAX=90.0 MAPLGMIN=-90 MAPBMAX=90.0 VCORLEV=0 NAN_BEYOND_RMAX=1 FORCEOUTPUT=1 >> jv2ts.log 2>&1\n"
else:  jv2ts = "jv2ts in=%s['%s'] v2hout=%s histlink=none TSTART='%s' TTOTAL='12m' TCHUNK='12m' MAPMMAX=5402 SINBDIVS=2160 LGSHIFT=3 CARRSTRETCH=1 MCORLEV=1 MAPRMAX=%s MAPLGMAX=90.0 MAPLGMIN=-90 MAPBMAX=90.0 VCORLEV=0 NAN_BEYOND_RMAX=1 FORCEOUTPUT=1 >> jv2ts.log 2>&1\n"

resizemappingmag_out = open(path_out+'3_resizemappingmag_%s.sh'%proj, 'w')
resizemappingmag_out.write('#!/bin/bash\n')
resizemappingmag = "resizemappingmag in=%s['%s'] out=%s nbin=3 >> resizemappingmag.log 2>&1\n"

trec_out = open(path_out+'trecs.txt', 'w')#

set_keys = "set_keys ds=%s[%s] %s=%s"

i = 0
for fname in fitsfiles:
    
    print('Processing %s...' %fname)
    
    # load with scaling to recognize blank cells -> necessary to prevent artifacts after resize
    fld = fits.open(path_out+fname)#, do_not_scale_image_data=True) 
    trec = fld[1].header['T_REC']
    fld.close()
    
    setinfo_out.write('\necho %s' %set_info %(data_series_m720s, trec, fname))
    setinfo_out.write(set_info %(data_series_m720s, trec, fname))
    
    file = fits.open(path_out+fname)[1]
    jv2ts_out.write('\n\necho %s' %jv2ts %(data_series_m720s, trec, data_series_jv2ts, trec, maprmax))
    jv2ts_out.write(jv2ts %(data_series_m720s, trec, data_series_jv2ts, trec, maprmax))
    jv2ts_out.write(set_keys %(data_series_jv2ts, trec, "CAR_ROT",  file.header['CAR_ROT2']))
    
    resizemappingmag_out.write('\necho %s' %resizemappingmag %(data_series_jv2ts, trec, data_series_remap))
    resizemappingmag_out.write(resizemappingmag %(data_series_jv2ts, trec, data_series_remap)) 
    
    trec_out.write("%s\n"%trec)
    i += 1

setinfo_out.write('\necho "done"')
jv2ts_out.write('\necho "done"')
resizemappingmag_out.write('\necho "done"')
    
setinfo_out.close()
jv2ts_out.close()
resizemappingmag_out.close()
trec_out.close()
print('done')

Processing solo_L2_phi-fdt-blos_20220603T031008_V202307311648_0246030501_drms.fits...
Processing solo_L2_phi-fdt-blos_20220603T150009_V202307311648_0246030503_drms.fits...
Processing solo_L2_phi-fdt-blos_20220604T030008_V202307311648_0246040501_drms.fits...
Processing solo_L2_phi-fdt-blos_20220604T150009_V202307311648_0246040503_drms.fits...
Processing solo_L2_phi-fdt-blos_20220605T030008_V202307311648_0246050501_drms.fits...
Processing solo_L2_phi-fdt-blos_20220605T150009_V202307311648_0246050503_drms.fits...
Processing solo_L2_phi-fdt-blos_20220606T031009_V202307311648_0246060501_drms.fits...
Processing solo_L2_phi-fdt-blos_20220606T073009_V202307311648_0246060502_drms.fits...
Processing solo_L2_phi-fdt-blos_20220607T030009_V202307311648_0246070501_drms.fits...
Processing solo_L2_phi-fdt-blos_20220607T150008_V202307311648_0246070503_drms.fits...
Processing solo_L2_phi-fdt-blos_20220607T210009_V202307311648_0246070504_drms.fits...
Processing solo_L2_phi-fdt-blos_20220608T030008_V20230

# Verification

In [44]:
#TODO OTUPUT PHI TIMESTRING

In [1]:
remap1 = fits.open('/SUM35/D281475007199139/S00000/Ml.fits')[0]
remap2 = fits.open('/SUM45/D281475007200085/S00000/Ml.fits')[0]
remap3 = fits.open('/SUM36/D281475007201224/S00000/Ml.fits')[0]
remap4 = fits.open('/SUM45/D281475007200466/S00000/Ml.fits')[0]
remap5 = fits.open('/SUM47/D281475007201234/S00000/Ml.fits')[0]
remap6 = fits.open('/SUM47/D281475007200825/S00000/Ml.fits')[0]
remap7 = fits.open('/SUM37/D281475007197791/S00000/Ml.fits')[0]
remap8 = fits.open('/SUM46/D281475007197940/S00000/Ml.fits')[0]
remap9 = fits.open('/SUM44/D281475007199498/S00000/Ml.fits')[0]
remap10 = fits.open('/SUM44/D281475007199317/S00000/Ml.fits')[0]
remap11 = fits.open('/SUM37/D281475007200105/S00000/Ml.fits')[0]
remap12 = fits.open('/SUM37/D281475007197502/S00000/Ml.fits')[0]

In [5]:
remap1_  = fits.open('/SUM47/D281475007164455/S00000/Ml.fits')[0]
remap2_  = fits.open('/SUM44/D281475007166624/S00000/Ml.fits')[0]
remap3_  = fits.open('/SUM46/D281475007166634/S00000/Ml.fits')[0]
remap4_  = fits.open('/SUM36/D281475007166654/S00000/Ml.fits')[0]
remap5_  = fits.open('/SUM44/D281475007166664/S00000/Ml.fits')[0]
remap6_  = fits.open('/SUM44/D281475007166674/S00000/Ml.fits')[0]
remap7_  = fits.open('/SUM34/D281475007166684/S00000/Ml.fits')[0]
remap8_  = fits.open('/SUM34/D281475007165485/S00000/Ml.fits')[0]
remap9_  = fits.open('/SUM34/D281475007166704/S00000/Ml.fits')[0]
remap10_ = fits.open('/SUM46/D281475007164436/S00000/Ml.fits')[0]
remap11_ = fits.open('/SUM37/D281475007159933/S00000/Ml.fits')[0]
remap12_ = fits.open('/SUM47/D281475007165445/S00000/Ml.fits')[0]

In [6]:
%matplotlib widget

In [11]:
plt.figure()
plt.imshow(remap7.data,vmin=-1500, vmax=1500, cmap='hmimag')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [ ]:
hmi = fits.open('../output/data/phi/feb2021/HMI_equivalent/hmi.M_720s.20210219_090000_TAI.3.magnetogram.fits')

In [ ]:
hmi[1].header['WAVELNTH']

In [ ]:
l2drms.header

# Development

## CR2240 VERSION

In [ ]:
path = "../output/data/phi/feb2021_rev02/"
dbpath = "/data/solo/phi/data/fmdb/l1/%s/%s"

tmp = 0
files = os.listdir(path)
fitsfiles = [file for file in files if file.endswith(".fits") or file.endswith(".fits.gz")]

remap = True 

for file in fitsfiles:
    l2 = fits.open(path+file)
  
    print("Processing %s ..." %file)
    
    # David's L2 files were processed with an old header version. 
    # Find respective UPDATED L1 files and copy the missing keywords
    # L1 parent FILENAME is outdated. Find source file with updated processing
    
    # solo_L1_phi-fdt-ilam_obsdateTobstime_processingdate_noclue.fits.gz
    # solo_L1_phi-fdt-ilam_20210205T200002_V202108301608C_0142050411.fits.gz
    
    # Extract outdated filename
    date = l2[0].header['FILENAME'][21:29] # eg '20210219'
    date_path = "%s-%s-%s" %(date[:4], date[4:6], date[6:8])

    # Find updated source file from observation date/time of outdated source file
    tmpfiles = os.listdir(dbpath %(date_path, ""))
    for tmpfile in tmpfiles:
        if l2[0].header['FILENAME'][:37] in tmpfile: # FILENAME[:37] eg: 'solo_L1_phi-fdt-ilam_20210205T200002_'
            break
    
    prim = fits.PrimaryHDU()
    l2drms = fits.CompImageHDU(data=l2[0].data.astype(np.int32))

    # open updated source file
    l1 = fits.open(dbpath %(date_path, tmpfile))

    l2drms.header.append(('', '', ''), end=True)
    l2drms.header.append(('', '  / HMI Compatibility', ''), end=True)
    
    # TODO TEMPORARY: Fix CAR_ROT bug
    if l2[0].header['CAR_ROT'] == 2239: 
        l2[0].header['CAR_ROT'] = 2240
        
    # T_REC / T_OBS
    #DATE-AVG= '2021-02-05T20:00:45.616' / [UTC] Average time of observation     
    #T_OBS   = '2021.02.28_07:11:55.437_TAI' / [TAI] nominal time 
    #T_REC   = '2021.02.28_07:12:00.000_TAI' / [TAI] Slot time    
    # Conversion for date format / UTC to TAI / round to the next 12 minute slot for T_REC
    trec = datetime.strptime(l2[0].header['DATE-AVG'],   "%Y-%m-%dT%H:%M:%S.%f")
    tobs = datetime.strptime(l2[0].header['DATE-AVG'],   "%Y-%m-%dT%H:%M:%S.%f")
    
    #utc2tai = timedelta(0, 37)                 # use for utc2tai conversion
    #trec = trec + utc2tai                      # use for utc2tai conversion
    trec = trec.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    tobs = tobs.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    
    # T_REC Interpolation    
    crln_obs = float(l2[0].header['CRLN_OBS'])
    car_rot  = int(l2[0].header['CAR_ROT'])
    
    if car_rot != tmp:
        # only do this loop at the beginning of each CAR_ROT
        crln_hmi, dt_hmi, t0, car_rot = prep_hmi_interp(crln_obs, car_rot, remap=remap)
        trec_hmi = interp_phi2hmi(crln_obs, crln_hmi, dt_hmi, t0)
        
        #TODO
        # change prep_hmi_interp() to return t0 and t1 with a logic that t0 is always the previous HMI reference
        #hmi_past = interp_phi2hmi(crln_obs, crln_hmi, dt_hmi, t0)
        #hmi_futr = interp_phi2hmi(crln_obs, crln_hmi, dt_hmi, t1)
        # save both keywords for orientation
        # use as T_REC according to closest CAR_ROT
        
        tmp = car_rot
    else:
        trec_hmi = interp_phi2hmi(crln_obs, crln_hmi, dt_hmi, t0)

    l2drms.header.append(('T_REC', trec, ''), end=True)
    l2drms.header.append(('T_OBS', tobs, ''), end=True)  # required for JV2TS
    
    l2drms.header.append(('T_REC_EPOCH', '1993.01.01_00:00:00_TAI', 'Time of origin'), end=True)
    l2drms.header.append(('T_REC_STEP', 720.0,  'ts_eq step'), end=True)
    
    # these two keywords are redundant with T_REC and T_OBS
    l2drms.header.append(('TREC_HMI', trec_hmi, ''), end=True) # HMI interpolated T_REC
    l2drms.header.append(('TREC_PHI', tobs, ''),     end=True) # real PHI observation date as backup
    
    # DATE
    l2drms.header.append(('DATE', l2[0].header['DATE'], "Date and time of FITS file creation, in UTC, in ISO-8601 format 'yyyy-mm-ddThh:mm:ss.sss'"), end=True)
    
    # DATE-OBS
    # DATE-BEG= '2021-02-05T20:00:02.906' / [UTC] Start time of observation 
    # DATE-OBS= '2021-02-28T07:10:33.400' / [ISO] Observation date {DATE__OBS}   
    # Conversion from BEG to OBS. Conversion from UTC to ISO
    date_obs = datetime.strptime(l2[0].header['DATE-BEG'],   "%Y-%m-%dT%H:%M:%S.%f")
    date_obs = date_obs.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    l2drms.header.append(('DATE-OBS', date_obs, 'DATE-OBS = DATE-AVG - EXPTIME/2.0'), end=True)

    # CADENCE
    # Missing, directly copy from HMI as a dummy value or extract daily? cadence from filenames
    l2drms.header.append(('CADENCE', 720.0, '[seconds] Observation cadence - DUMMY VALUE'), end=True)
    
    # TELESCOP
    l2drms.header.append(('TELESCOP', l2[0].header['TELESCOP'], 'For PHI: SOLO/PHI/FDT or SOLO/PHI/HRT - For HMI: SDO/HMI'), end=True)
    
    # INSTRUME
    l2drms.header.append(('INSTRUME', l2[0].header['INSTRUME'], 'For PHI: PHI - For HMI: HMI_SIDE1, HMI_FRONT2, or HMI_COMBINED'), end=True)
    
    # WAVELNTH
    l2drms.header.append(('WAVELNTH', 6173.341, 'For PHI/HMI: 6173.3 Angstroms'), end=True)
    
    # QUALITY
    # Missing. add as dummy value = 0. Quality index of data used to genrate fd.B_720s (0x00000000) is good quality, (any nonzero value) should be investiaged in data documentation
    l2drms.header.append(('QUALITY', 0, 'Level 1.5 Quality - DUMMY VALUE'), end=True)

    # BUNIT
    l2drms.header.append(('BUNIT', l2[0].header['BUNIT'], 'BUNIT: physical units of each data segment'), end=True)
    
    # HISTORY
    # TODO FIX FILENAME ONCE WE HAVE GOOD L2 HEADERS
    l2drms.header.append(('HISTORY', 'DRMS compatible FITS created from %s' %l1[0].header['FILENAME'], 'History of data'), end=True)
    
    # COMMENT
    #l2drms.header.append(('COMMENT', l2[0].header['COMMENT'], 'Commentary on the data'), end=True)
    
    # CTYPE1
    l2drms.header.append(('CTYPE1', l2[0].header['CTYPE1'], 'CTYPE1: HPLN-TAN (SOLARX)'), end=True)
    
    # CTYPE2
    l2drms.header.append(('CTYPE2', l2[0].header['CTYPE2'], 'CTYPE2: HPLN-TAN (SOLARY)'), end=True)
    
    #CRPIX1
    l2drms.header.append(('CRPIX1', l2[0].header['CRPIX1'], 'CRPIX1: location of the Sun center in CCD x direction'), end=True)
    
    #CRPIX2
    l2drms.header.append(('CRPIX2', l2[0].header['CRPIX2'], 'CRPIX2: location of the Sun center in CCD y direction'), end=True)
    
    #CRVAL1
    l2drms.header.append(('CRVAL1', l2[0].header['CRVAL1'], 'CRVAL1: x origin - center of the solar disk'), end=True)
    
    #CRVAL2
    l2drms.header.append(('CRVAL2', l2[0].header['CRVAL2'], 'CRVAL2: y origin - center of the solar disk'), end=True)
    
    #CDELT1
    l2drms.header.append(('CDELT1', l2[0].header['CDELT1'], 'Image scale in the x direction'), end=True)
    
    #CDELT2
    l2drms.header.append(('CDELT2', l2[0].header['CDELT2'], 'Image scale in the y direction'), end=True)
    
    #CUNIT1
    l2drms.header.append(('CUNIT1', l2[0].header['CUNIT1'], 'CUNIT1: arcsec'), end=True)
    
    #CUNIT2
    l2drms.header.append(('CUNIT2', l2[0].header['CUNIT2'], 'CUNIT2: arcsec'), end=True)
    
    # CROTA 
    # CROTA2
    # CROTA was renamed to CROTA2 in Dietmar's current L2 header. Rename here for now. CHECK IF THE ROTATION MAKES SENSE AFTER THE PROJECTION by comparing the projections of equal CRLN_OBS in PHI and HMI
    l2drms.header.append(('CROTA2', l2[0].header['CROTA'], '[deg] Rotation angle'), end=True)

    #CRDER1
    #l2drms.header.append(('CRDER1', l1[0].header['CRDER1'], 'CRDER1: estimate of random error in coordinate x'), end=True)
    
    #CRDER2
    #l2drms.header.append(('CRDER2', l1[0].header['CRDER2'], 'CRDER2: estimate of random error in coordinate y'), end=True)
    
    #CSYSER1
    #l2drms.header.append(('CSYSER1', l2[0].header['CSYSER1'], 'CSYSER1: estimate of systematic error in coordinate x'), end=True)
    
    #CSYSER2
    #l2drms.header.append(('CSYSER2', l2[0].header['CSYSER2'], 'CSYSER2: estimate of systematic error in coordinate y'), end=True)
    
    #WCSNAME
    l2drms.header.append(('WCSNAME', l2[0].header['WCSNAME'], 'WCS system name'), end=True)
    
    #DSUN_OBS
    l2drms.header.append(('DSUN_OBS', l2[0].header['DSUN_OBS'], 'Distance from SDO to Sun center.'), end=True)
    
    #RSUN_REF
    l2drms.header.append(('RSUN_REF', l2[0].header['RSUN_REF'], 'Reference radius of the Sun: 696,000,000.0 m'), end=True)
    
    #CRLN_OBS
    l2drms.header.append(('CRLN_OBS', l2[0].header['CRLN_OBS'], 'Carrington longitude of HMI'), end=True)
    
    #CRLT_OBS
    l2drms.header.append(('CRLT_OBS', l2[0].header['CRLT_OBS'], 'Carrington latitude of HMI'), end=True)
    
    #CAR_ROT        
    l2drms.header.append(('CAR_ROT', car_rot, 'Carrington rotation number of CRLN_OBS'), end=True)
    
    # OBS_VW
    # OBS_VR
    # OBS_VN
    # Both are missing in the OLD HEADER (David's) but should't in the future. Fetch keywords from L1 data, which is the one listed in the processed FILENAME history keyword
    # eg.: FILENAME solo_L1_phi-fdt-ilam_20210205T200002_V202108301608C_0142050411.fits
    l2drms.header.append(('OBS_VR', l1[0].header['OBS_VR'], '[m/s] Radial velocity of S/C relative to Sun   '), end=True)
    l2drms.header.append(('OBS_VW', l1[0].header['OBS_VW'], '[m/s] Westward velocity of S/C relative to Sun '), end=True)
    l2drms.header.append(('OBS_VN', l1[0].header['OBS_VN'], '[m/s] Northward velocity of S/C relative to Sun'), end=True)

    # RSUN_ARC
    # RSUN_OBS
    # Same keyword. Save RSUN_ARC as RSUN_OBS for HMI
    l2drms.header.append(('RSUN_OBS', l2[0].header['RSUN_ARC'], '[arcsec] angular radius of Sun.'), end=True)

    # DATAVALS
    # Actual number of data values in images (pixels)
    l2drms.header.append(('DATAVALS', l2[0].header['NAXIS1']*l2[0].header['NAXIS2'], 'Actual number of data values in images'), end=True)

    # MISSVALS
    # Missing values: TOTVALS - DATAVALS
    l2drms.header.append(('MISSVALS', 0, 'Missing values: TOTVALS - DATAVALS'), end=True)
    
    # DATAMIN
    l2drms.header.append(('DATAMIN', l2[0].header['DATAMIN'], 'Minimum value from pixels within 99% of solar radius'), end=True)
    
    # DATAMAX
    l2drms.header.append(('DATAMAX', l2[0].header['DATAMAX'], 'Maximum value from pixels within 99% of solar radius'), end=True)
    
   
    hdul = fits.HDUList([prim, l2drms])
    #hdul.writeto(path+'drms/%s_drms.fits' %file[:-5], overwrite=True)
    l1.close()
    break
print('Done')

#ef make_fits(phi, hmi, dtype):
#   # phi ... image
#   # hmi ... image

#   prim = fits.PrimaryHDU(data=hmi[0].data, header=hmi[0].header)
#   cimg = fits.CompImageHDU(data=phi.astype(dtype), header=hmi[1].header[:-2]) # -2 skips old checksum and datasum
#   hdul = fits.HDUList([prim, cimg])
#       
#   return hdul

## Misc

In [ ]:
%matplotlib widget

In [ ]:
#mps_loeschl.Ml_hiresmap_720s_test[2021.02.19_12:30:03_TAI]	/SUM36/D281475007162730/S00000
hires = fits.open("/SUM36/D281475007162730/S00000/Ml.fits")

In [ ]:
hires[0].header

In [ ]:
import sunpy.map
from matplotlib import pyplot as plt

fig, ax = plt.subplots()
ax.imshow(hires[0].data, vmin=-1500, vmax=1500, cmap='hmimag', origin='lower')
plt.show()

## Carrington Date Calculator

In [ ]:
from sunpy.coordinates.sun import carrington_rotation_time
import astropy.units as u
from astropy.time import Time

In [ ]:
carrington_rotation_time(2250)

In [21]:
carr_time = (Time(l2[0].header['DATE-AVG'], format='isot', scale='utc') - carrington_rotation_time(l2[0].header['CAR_ROT'])).value 
carr_dur = (carrington_rotation_time(l2[0].header['CAR_ROT']+1) - carrington_rotation_time(l2[0].header['CAR_ROT'])).value
crln_rot = carr_time / carr_dur  * 360
crln_rot

-67.08202682840073

In [22]:
carr_time

-5.091798414230175

In [24]:
carrington_rotation_time(l2[0].header['CAR_ROT']).value 
Time(l2[0].header['DATE-AVG'], format='isot', scale='utc')

'2021-02-18 14:42:57.043'

In [42]:
carrington_rotation_time(2240)

<Time object: scale='utc' format='iso' value=2021-01-22 06:31:17.266>

In [43]:
carrington_rotation_time(2241)

<Time object: scale='utc' format='iso' value=2021-02-18 14:42:57.043>

## T_REC Mapping

In [20]:
def get_drms_keywords(inRecs, input_ds):

    #inRecs = "2014.05.12_12:00:00_TAI, 2014.05.13_00:00:00_TAI, 2014.05.13_12:00:00_TAI, 2014.05.14_00:00:00_TAI" # input argument
    show_info = 'show_info %s["%s"] key="T_REC,CRLN_OBS,CAR_ROT"'
    
    #-P for path and -A for segment
    si_out = subprocess.check_output(show_info %(input_ds, inRecs) , shell=True)[:-1].decode("utf-8")
    raw = si_out.split('\n')

    formatted = [] 
    drms_param = []
    
    nRecs = 0
    keys = raw[0].split('\t')
    
    for line in raw[1:]:  
        formatted = line.split('\t') # [CALVER64, T_REC, QUALITY, FDRADIAL, CARSTRCH, DIFROT_A, DIFROT_B, DIFROT_C, CRVAL1, CRLN_OBS, CAR_ROT, MAPLGMAX, MAPLGMIN, I_DREC]

        dict_tmp = {}

        for i, key in enumerate(keys):
            
            if key == "magnetogram" or key == 'Ml':
                key = "PATH"
                
            if formatted[i].strip() == "InvalidKeyname":
                dict_tmp[key] = 0
            else:
                dict_tmp[key] = formatted[i]
   
        drms_param.append(dict_tmp)
        nRecs += 1

    return drms_param, nRecs


In [282]:
hmi_times = "2021.02.19_09:00:00_TAI,2021.02.22_10:48:00_TAI,2021.02.23_06:48:00_TAI,2021.02.24_02:36:00_TAI,2021.02.24_22:36:00_TAI,2021.02.25_18:36:00_TAI,2021.02.26_14:36:00_TAI,2021.02.28_07:12:00_TAI"
phi_times = "2021.02.05_20:00:02_TAI,2021.02.09_12:30:03_TAI,2021.02.10_12:30:03_TAI,2021.02.11_12:30:02_TAI,2021.02.12_12:30:03_TAI,2021.02.13_12:30:02_TAI,2021.02.14_12:30:03_TAI,2021.02.16_12:30:02_TAI"

## Manual HMI equivalent

In [283]:
hmi_data, _= get_drms_keywords(hmi_times,"mps_loeschl.Ml_remap_720s")
phi_data, _= get_drms_keywords(phi_times,"mps_loeschl.Ml_remap_720s_test")

In [284]:
# this version requires manual selection of the HMI data
dt_hmi   = np.array([])
trec_hmi = np.array([])
crln_hmi = np.array([])

# Prepare the HMI data for interpolation
t0 = carrington_rotation_time(hmi_data[i]['CAR_ROT'])

for line in hmi_data:    
    trec_hmi = np.append(trec_hmi, datetime.strptime(line['T_REC'], "%Y.%m.%d_%H:%M:%S_TAI"))
    crln_hmi = np.append(crln_hmi, float(line['CRLN_OBS']))
    
    dt_hmi   = np.append(dt_hmi, ((trec_hmi[-1] - t0.datetime).days +(trec_hmi[-1] - t0.datetime).seconds/(3600*24)))
    
for i, data in enumerate(phi_data):       
    t_interp = timedelta(days=np.interp(phi_data[i]['CRLN_OBS'], crln_hmi, dt_hmi, period=360))
    trec_phi = t0.datetime+t_interp
    phi_data[i]['T_REC_HMI'] = trec_phi.strftime("%Y.%m.%d_%H:%M:%S_TAI")

In [285]:
phi_data

[{'T_REC': '2021.02.05_20:00:45_TAI',
  'CRLN_OBS': '349.878540',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.19_09:10:54_TAI'},
 {'T_REC': '2021.02.09_12:30:45_TAI',
  'CRLN_OBS': '309.480194',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.22_10:48:39_TAI'},
 {'T_REC': '2021.02.10_12:30:45_TAI',
  'CRLN_OBS': '298.545929',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.23_06:42:54_TAI'},
 {'T_REC': '2021.02.11_12:30:45_TAI',
  'CRLN_OBS': '287.611237',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.24_02:37:27_TAI'},
 {'T_REC': '2021.02.12_12:30:45_TAI',
  'CRLN_OBS': '276.671143',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.24_22:33:55_TAI'},
 {'T_REC': '2021.02.13_12:30:45_TAI',
  'CRLN_OBS': '265.720734',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.25_18:32:23_TAI'},
 {'T_REC': '2021.02.14_12:30:45_TAI',
  'CRLN_OBS': '254.755127',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.26_14:32:06_TAI'},
 {'T_REC': '2021.02.16_12:30:45_TAI',
  'CRLN_OBS': '232.760178',
  'CAR_ROT': '2241',
  '

## Automatic HMI equivalent

In [294]:
hmi_times = "2021.02.19_09:00:00_TAI,2021.02.22_10:48:00_TAI,2021.02.23_06:48:00_TAI,2021.02.24_02:36:00_TAI,2021.02.24_22:36:00_TAI,2021.02.25_18:36:00_TAI,2021.02.26_14:36:00_TAI,2021.02.28_07:12:00_TAI"
phi_times = "2021.02.05_20:00:02_TAI,2021.02.09_12:30:03_TAI,2021.02.10_12:30:03_TAI,2021.02.11_12:30:02_TAI,2021.02.12_12:30:03_TAI,2021.02.13_12:30:02_TAI,2021.02.14_12:30:03_TAI,2021.02.16_12:30:02_TAI"

In [317]:
# find HMI equivalent from PHI meta data
# requries manual selection of PHI data from a SINGLE CAR_ROT
#hmi_data, _= get_drms_keywords(hmi_times,"mps_loeschl.Ml_remap_720s")

phi_data, _= get_drms_keywords(phi_times,"mps_loeschl.Ml_remap_720s_test")

dt_hmi   = np.array([])
trec_hmi = np.array([])
crln_hmi = np.array([])

# Prepare the HMI data for interpolation

remap = False
for i, data in enumerate(phi_data):  
    if remap and float(phi_data[i]['CRLN_OBS']) > 180:
        phi_data[i]['CAR_ROT'] = int(phi_data[i]['CAR_ROT']) - 1
    else: 
        phi_data[i]['CAR_ROT'] = int(phi_data[i]['CAR_ROT']) 

t0 = carrington_rotation_time(phi_data[0]['CAR_ROT'])
t1 = carrington_rotation_time(phi_data[0]['CAR_ROT']+1)

hmi_times = "%s-%s" %(t0.datetime.strftime("%Y.%m.%d_%H:%M:%S_TAI"), t1.datetime.strftime("%Y.%m.%d_%H:%M:%S_TAI"))
hmi_data, n = get_drms_keywords(hmi_times, "hmi.m_720s") #"mps_loeschl.Ml_remap_720s")

print('Mapping PHI to CR %s in HMI period %s...' %(phi_data[0]['CAR_ROT'], hmi_times))

for line in hmi_data:    
    trec_hmi = np.append(trec_hmi, datetime.strptime(line['T_REC'], "%Y.%m.%d_%H:%M:%S_TAI"))
    crln_hmi = np.append(crln_hmi, float(line['CRLN_OBS']))
    dt_hmi   = np.append(dt_hmi, ((trec_hmi[-1] - t0.datetime).days +(trec_hmi[-1] - t0.datetime).seconds/(3600*24)))    
    
for i, data in enumerate(phi_data):       
    t_interp = timedelta(days=np.interp(phi_data[i]['CRLN_OBS'], crln_hmi, dt_hmi, period=360))
    trec_phi = t0.datetime+t_interp
    phi_data[i]['TREC_HMI'] = trec_phi.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    phi_data[i]['TREC_PHI'] = phi_data[i]['T_REC']
    

print('done')

Mapping PHI to CR 2241 in HMI period 2021.02.18_14:42:57_TAI-2021.03.17_22:31:37_TAI...
done


## Automatic HMI equivalent without CAR_ROT dependency

In [294]:
hmi_times = "2021.02.19_09:00:00_TAI,2021.02.22_10:48:00_TAI,2021.02.23_06:48:00_TAI,2021.02.24_02:36:00_TAI,2021.02.24_22:36:00_TAI,2021.02.25_18:36:00_TAI,2021.02.26_14:36:00_TAI,2021.02.28_07:12:00_TAI"
phi_times = "2021.02.05_20:00:02_TAI,2021.02.09_12:30:03_TAI,2021.02.10_12:30:03_TAI,2021.02.11_12:30:02_TAI,2021.02.12_12:30:03_TAI,2021.02.13_12:30:02_TAI,2021.02.14_12:30:03_TAI,2021.02.16_12:30:02_TAI"

In [317]:
# find HMI equivalent from PHI meta data
# requries manual selection of PHI data from a SINGLE CAR_ROT
#hmi_data, _= get_drms_keywords(hmi_times,"mps_loeschl.Ml_remap_720s")

phi_data, _= get_drms_keywords(phi_times,"mps_loeschl.Ml_remap_720s_test")

dt_hmi   = np.array([])
trec_hmi = np.array([])
crln_hmi = np.array([])

# Prepare the HMI data for interpolation

remap = False
for i, data in enumerate(phi_data):  
    if remap and float(phi_data[i]['CRLN_OBS']) > 180:
        phi_data[i]['CAR_ROT'] = int(phi_data[i]['CAR_ROT']) - 1
    else: 
        phi_data[i]['CAR_ROT'] = int(phi_data[i]['CAR_ROT']) 

t0 = carrington_rotation_time(phi_data[0]['CAR_ROT'])
t1 = carrington_rotation_time(phi_data[0]['CAR_ROT']+1)

hmi_times = "%s-%s" %(t0.datetime.strftime("%Y.%m.%d_%H:%M:%S_TAI"), t1.datetime.strftime("%Y.%m.%d_%H:%M:%S_TAI"))
hmi_data, n = get_drms_keywords(hmi_times, "hmi.m_720s") #"mps_loeschl.Ml_remap_720s")

print('Mapping PHI to CR %s in HMI period %s...' %(phi_data[0]['CAR_ROT'], hmi_times))

for line in hmi_data:    
    trec_hmi = np.append(trec_hmi, datetime.strptime(line['T_REC'], "%Y.%m.%d_%H:%M:%S_TAI"))
    crln_hmi = np.append(crln_hmi, float(line['CRLN_OBS']))
    dt_hmi   = np.append(dt_hmi, ((trec_hmi[-1] - t0.datetime).days +(trec_hmi[-1] - t0.datetime).seconds/(3600*24)))    
    
for i, data in enumerate(phi_data):       
    t_interp = timedelta(days=np.interp(phi_data[i]['CRLN_OBS'], crln_hmi, dt_hmi, period=360))
    trec_phi = t0.datetime+t_interp
    phi_data[i]['TREC_HMI'] = trec_phi.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    phi_data[i]['TREC_PHI'] = phi_data[i]['T_REC']
    

print('done')

Mapping PHI to CR 2241 in HMI period 2021.02.18_14:42:57_TAI-2021.03.17_22:31:37_TAI...
done


### Pipeline Integration

#### New Version

In [11]:
def calc_trec(crln_obs, car_rot, verbose=False):
    # helper function to prepare hmi data for interp_phi2hmi() interpolation

    # remap and >180 means that HMI_PAST is the previous CAR_ROT and HMI_FUTR is the current CAR_ROT
    # remap and <180 means that HMI_FUTR is the current CAR_ROT and HMI_PAST is the previous CAR_ROT
    print(crln_obs, car_rot)
    if crln_obs > 180:
        # t defined by when HMI sees it (future/past)
        #t0 = carrington_rotation_time(car_rot-1) # past / current
        #t1 = carrington_rotation_time(car_rot)   # future
        
        t0 = carrington_rotation_time(car_rot-1) # past / current CAR_ROT
        t1 = carrington_rotation_time(car_rot)   # future
        t2 = carrington_rotation_time(car_rot+1) # future end point
        
        trec_hmi = interp_phi2hmi(crln_obs, t0, t1, verbose)
        hmi_next = interp_phi2hmi(crln_obs, t1, t2)
        hmi_prev = trec_hmi
        car_rot -= 1
        
    else:
        t0 = carrington_rotation_time(car_rot-1) # past  
        t1 = carrington_rotation_time(car_rot)   # future / current
        t2 = carrington_rotation_time(car_rot+1) # future end point

        trec_hmi = interp_phi2hmi(crln_obs, t1, t2, verbose)
        hmi_prev = interp_phi2hmi(crln_obs, t0, t1)
        hmi_next = trec_hmi
        
    return trec_hmi, hmi_prev, hmi_next, car_rot


def interp_phi2hmi(crln_obs, t0, t1, verbose=False):
    # Interpolate T_REC of PHI CRLN_OBS onto HMI CRLN_OBS
    dt_hmi   = np.array([])
    trec_hmi = np.array([])
    crln_hmi = np.array([])
    
    hmi_times = "%s-%s" %(t0.datetime.strftime("%Y.%m.%d_%H:%M:%S_TAI"), t1.datetime.strftime("%Y.%m.%d_%H:%M:%S_TAI"))
    hmi_data, n = get_drms_keywords(hmi_times, "hmi.m_720s") #"mps_loeschl.Ml_remap_720s")

    if verbose: print('Mapping PHI to CR %s in HMI period %s...' %(car_rot, hmi_times))

    for line in hmi_data:    
        trec_hmi = np.append(trec_hmi, datetime.strptime(line['T_REC'], "%Y.%m.%d_%H:%M:%S_TAI"))
        crln_hmi = np.append(crln_hmi, float(line['CRLN_OBS']))
        dt_hmi   = np.append(dt_hmi, ((trec_hmi[-1] - t0.datetime).days +(trec_hmi[-1] - t0.datetime).seconds/(3600*24)))    
        
    t_interp = timedelta(days=np.interp(crln_obs, crln_hmi, dt_hmi, period=360))
    trec_phi = t0.datetime+t_interp
    trec_hmi = trec_phi.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    
    return trec_hmi
    

In [382]:
#TODO FOR CR2240: include the l1 keywords updates again

path = "../output/data/phi/feb2021/"
dbpath = "/data/solo/phi/data/fmdb/l1/%s/%s"

tmp = 0
files = os.listdir(path)
fitsfiles = [file for file in files if file.endswith(".fits")]

remap = True 

for file in fitsfiles:
    l2 = fits.open(path+file)
  
    print("Processing %s ..." %file)
    
    # David's L2 files were processed with an old header version. 
    # Find respective UPDATED L1 files and copy the missing keywords
    # L1 parent FILENAME is outdated. Find source file with updated processing
    
    # solo_L1_phi-fdt-ilam_obsdateTobstime_processingdate_noclue.fits.gz
    # solo_L1_phi-fdt-ilam_20210205T200002_V202108301608C_0142050411.fits.gz
    
    # Extract outdated filename
    date = l2[0].header['FILENAME'][21:29] # eg '20210219'
    date_path = "%s-%s-%s" %(date[:4], date[4:6], date[6:8])

    # Find updated source file from observation date/time of outdated source file
    tmpfiles = os.listdir(dbpath %(date_path, ""))
    for tmpfile in tmpfiles:
        if l2[0].header['FILENAME'][:37] in tmpfile: # FILENAME[:37] eg: 'solo_L1_phi-fdt-ilam_20210205T200002_'
            break
    
    prim = fits.PrimaryHDU()
    l2drms = fits.CompImageHDU(data=l2[0].data.astype(np.int32))

    # open updated source file
    #l1 = fits.open(dbpath %(date_path, tmpfile))

    l2drms.header.append(('', '', ''), end=True)
    l2drms.header.append(('', '  / HMI Compatibility', ''), end=True)
    
    # TODO TEMPORARY: Fix CAR_ROT bug
    if l2[0].header['CAR_ROT'] == 2239: 
        l2[0].header['CAR_ROT'] = 2240
        
    # T_REC / T_OBS
    #DATE-AVG= '2021-02-05T20:00:45.616' / [UTC] Average time of observation     
    #T_OBS   = '2021.02.28_07:11:55.437_TAI' / [TAI] nominal time 
    #T_REC   = '2021.02.28_07:12:00.000_TAI' / [TAI] Slot time    
    # Conversion for date format / UTC to TAI / round to the next 12 minute slot for T_REC
    #trec = datetime.strptime(l2[0].header['DATE-AVG'],   "%Y-%m-%dT%H:%M:%S.%f")
    tobs = datetime.strptime(l2[0].header['DATE-AVG'],   "%Y-%m-%dT%H:%M:%S.%f")
    
    #utc2tai = timedelta(0, 37)                 # use for utc2tai conversion
    #trec = trec + utc2tai                      # use for utc2tai conversion
    #trec = trec.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    tobs = tobs.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    
    # T_REC Interpolation       
    trec, hmi_prev, hmi_next, car_rot = calc_trec(float(l2[0].header['CRLN_OBS']), int(l2[0].header['CAR_ROT']), verbose=False)

    l2drms.header.append(('T_REC', trec, ''), end=True)
    l2drms.header.append(('T_OBS', tobs, ''), end=True)  # required for JV2TS
    
    l2drms.header.append(('T_REC_EPOCH', '1993.01.01_00:00:00_TAI', 'Time of origin'), end=True)
    l2drms.header.append(('T_REC_STEP', 720.0,  'ts_eq step'), end=True)
    
    # these two keywords are redundant with T_REC and T_OBS
    l2drms.header.append(('HMI_PREV', hmi_prev, ''), end=True) # HMI interpolated T_REC
    l2drms.header.append(('HMI_NEXT', hmi_next, ''), end=True) # real PHI observation date as backup
    
    # DATE
    l2drms.header.append(('DATE', l2[0].header['DATE'], "Date and time of FITS file creation, in UTC, in ISO-8601 format 'yyyy-mm-ddThh:mm:ss.sss'"), end=True)
    
    # DATE-OBS
    # DATE-BEG= '2021-02-05T20:00:02.906' / [UTC] Start time of observation 
    # DATE-OBS= '2021-02-28T07:10:33.400' / [ISO] Observation date {DATE__OBS}   
    # Conversion from BEG to OBS. Conversion from UTC to ISO
    date_obs = datetime.strptime(l2[0].header['DATE-BEG'],   "%Y-%m-%dT%H:%M:%S.%f")
    date_obs = date_obs.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    l2drms.header.append(('DATE-OBS', date_obs, 'DATE-OBS = DATE-AVG - EXPTIME/2.0'), end=True)

    # CADENCE
    # Missing, directly copy from HMI as a dummy value or extract daily? cadence from filenames
    l2drms.header.append(('CADENCE', 720.0, '[seconds] Observation cadence - DUMMY VALUE'), end=True)
    
    # TELESCOP
    l2drms.header.append(('TELESCOP', l2[0].header['TELESCOP'], 'SOLO/PHI/FDT, SOLO/PHI/HRT, HMI: SDO/HMI'), end=True)
    
    # INSTRUME
    l2drms.header.append(('INSTRUME', l2[0].header['INSTRUME'], 'PHI, HMI_SIDE1, HMI_FRONT2, HMI_COMBINED'), end=True)
    
    # WAVELNTH
    l2drms.header.append(('WAVELNTH', 6173.341, 'For PHI/HMI: 6173.3 Angstroms'), end=True)
    
    # QUALITY
    # Missing. add as dummy value = 0. Quality index of data used to genrate fd.B_720s (0x00000000) is good quality, (any nonzero value) should be investiaged in data documentation
    l2drms.header.append(('QUALITY', 0, 'Level 1.5 Quality - DUMMY VALUE'), end=True)

    # BUNIT
    l2drms.header.append(('BUNIT', l2[0].header['BUNIT'], 'BUNIT: physical units of each data segment'), end=True)
    
    # HISTORY
    # TODO FIX FILENAME ONCE WE HAVE GOOD L2 HEADERS
    l2drms.header.append(('HISTORY', 'DRMS compatible FITS created from %s' %l1[0].header['FILENAME'], 'History of data'), end=True)
    
    # COMMENT
    #l2drms.header.append(('COMMENT', l2[0].header['COMMENT'], 'Commentary on the data'), end=True)
    
    # CTYPE1
    l2drms.header.append(('CTYPE1', l2[0].header['CTYPE1'], 'CTYPE1: HPLN-TAN (SOLARX)'), end=True)
    
    # CTYPE2
    l2drms.header.append(('CTYPE2', l2[0].header['CTYPE2'], 'CTYPE2: HPLN-TAN (SOLARY)'), end=True)
    
    #CRPIX1
    l2drms.header.append(('CRPIX1', l2[0].header['CRPIX1'], 'CRPIX1: location of the Sun center in CCD x direction'), end=True)
    
    #CRPIX2
    l2drms.header.append(('CRPIX2', l2[0].header['CRPIX2'], 'CRPIX2: location of the Sun center in CCD y direction'), end=True)
    
    #CRVAL1
    l2drms.header.append(('CRVAL1', l2[0].header['CRVAL1'], 'CRVAL1: x origin - center of the solar disk'), end=True)
    
    #CRVAL2
    l2drms.header.append(('CRVAL2', l2[0].header['CRVAL2'], 'CRVAL2: y origin - center of the solar disk'), end=True)
    
    #CDELT1
    l2drms.header.append(('CDELT1', l2[0].header['CDELT1'], 'Image scale in the x direction'), end=True)
    
    #CDELT2
    l2drms.header.append(('CDELT2', l2[0].header['CDELT2'], 'Image scale in the y direction'), end=True)
    
    #CUNIT1
    l2drms.header.append(('CUNIT1', l2[0].header['CUNIT1'], 'CUNIT1: arcsec'), end=True)
    
    #CUNIT2
    l2drms.header.append(('CUNIT2', l2[0].header['CUNIT2'], 'CUNIT2: arcsec'), end=True)
    
    # CROTA 
    # CROTA2
    # CROTA was renamed to CROTA2 in Dietmar's current L2 header. Rename here for now. CHECK IF THE ROTATION MAKES SENSE AFTER THE PROJECTION by comparing the projections of equal CRLN_OBS in PHI and HMI
    l2drms.header.append(('CROTA2', l2[0].header['CROTA'], '[deg] Rotation angle'), end=True)

    #CRDER1
    #l2drms.header.append(('CRDER1', l1[0].header['CRDER1'], 'CRDER1: estimate of random error in coordinate x'), end=True)
    
    #CRDER2
    #l2drms.header.append(('CRDER2', l1[0].header['CRDER2'], 'CRDER2: estimate of random error in coordinate y'), end=True)
    
    #CSYSER1
    #l2drms.header.append(('CSYSER1', l2[0].header['CSYSER1'], 'CSYSER1: estimate of systematic error in coordinate x'), end=True)
    
    #CSYSER2
    #l2drms.header.append(('CSYSER2', l2[0].header['CSYSER2'], 'CSYSER2: estimate of systematic error in coordinate y'), end=True)
    
    #WCSNAME
    l2drms.header.append(('WCSNAME', l2[0].header['WCSNAME'], 'WCS system name'), end=True)
    
    #DSUN_OBS
    l2drms.header.append(('DSUN_OBS', l2[0].header['DSUN_OBS'], 'Distance from SDO to Sun center.'), end=True)
    
    #RSUN_REF
    l2drms.header.append(('RSUN_REF', l2[0].header['RSUN_REF'], 'Reference radius of the Sun: 696,000,000.0 m'), end=True)
    
    #CRLN_OBS
    l2drms.header.append(('CRLN_OBS', l2[0].header['CRLN_OBS'], 'Carrington longitude of HMI'), end=True)
    
    #CRLT_OBS
    l2drms.header.append(('CRLT_OBS', l2[0].header['CRLT_OBS'], 'Carrington latitude of HMI'), end=True)
    
    #CAR_ROT        
    l2drms.header.append(('CAR_ROT', l2[0].header['CAR_ROT'], 'Carrington rotation number of CRLN_OBS'), end=True)
    l2drms.header.append(('CAR_ROT2', car_rot, 'Carrington rotation number of synoptic map'), end=True)
    
    # OBS_VW
    # OBS_VR
    # OBS_VN
    # Both are missing in the OLD HEADER (David's) but should't in the future. Fetch keywords from L1 data, which is the one listed in the processed FILENAME history keyword
    # eg.: FILENAME solo_L1_phi-fdt-ilam_20210205T200002_V202108301608C_0142050411.fits
    l2drms.header.append(('OBS_VR', l1[0].header['OBS_VR'], '[m/s] Radial velocity of S/C relative to Sun   '), end=True)
    l2drms.header.append(('OBS_VW', l1[0].header['OBS_VW'], '[m/s] Westward velocity of S/C relative to Sun '), end=True)
    l2drms.header.append(('OBS_VN', l1[0].header['OBS_VN'], '[m/s] Northward velocity of S/C relative to Sun'), end=True)

    # RSUN_ARC
    # RSUN_OBS
    # Same keyword. Save RSUN_ARC as RSUN_OBS for HMI
    l2drms.header.append(('RSUN_OBS', l2[0].header['RSUN_ARC'], '[arcsec] angular radius of Sun.'), end=True)

    # DATAVALS
    # Actual number of data values in images (pixels)
    l2drms.header.append(('DATAVALS', l2[0].header['NAXIS1']*l2[0].header['NAXIS2'], 'Actual number of data values in images'), end=True)

    # MISSVALS
    # Missing values: TOTVALS - DATAVALS
    l2drms.header.append(('MISSVALS', 0, 'Missing values: TOTVALS - DATAVALS'), end=True)
    
    # DATAMIN
    l2drms.header.append(('DATAMIN', l2[0].header['DATAMIN'], 'Minimum value from pixels within 99% of solar radius'), end=True)
    
    # DATAMAX
    l2drms.header.append(('DATAMAX', l2[0].header['DATAMAX'], 'Maximum value from pixels within 99% of solar radius'), end=True)
    
   
    hdul = fits.HDUList([prim, l2drms])
    #hdul.writeto(path+'drms/%s_drms.fits' %file[:-5], overwrite=True)
    l1.close()
    break
print('Done')

#ef make_fits(phi, hmi, dtype):
#   # phi ... image
#   # hmi ... image

#   prim = fits.PrimaryHDU(data=hmi[0].data, header=hmi[0].header)
#   cimg = fits.CompImageHDU(data=phi.astype(dtype), header=hmi[1].header[:-2]) # -2 skips old checksum and datasum
#   hdul = fits.HDUList([prim, cimg])
#       
#   return hdul

Processing solo_L2_phi-fdt-blos_20210205T200002_V202107130921C_0142050411.fits ...
Done


In [383]:
l2drms.header

XTENSION= 'IMAGE   '           / Image extension                                
BITPIX  =                   32 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 1280                                                  
NAXIS2  =                 1280                                                  
PCOUNT  =                    0 / number of parameters                           
GCOUNT  =                    1 / number of groups                               
                                                                                
          / HMI Compatibility                                                   
T_REC   = '2021.01.23_00:57:34_TAI'                                             
T_OBS   = '2021.02.05_20:00:45_TAI'                                             
HIERARCH T_REC_EPOCH = '1993.01.01_00:00:00_TAI' / Time of origin               
HIERARCH T_REC_STEP = 720.0 

#### Old Version

In [370]:
def prep_hmi_interp(crln_obs, car_rot, remap=False, verbose=False):
    # helper function to prepare hmi data for interp_phi2hmi() interpolation
    
    dt_hmi   = np.array([])
    trec_hmi = np.array([])
    crln_hmi = np.array([])
    
    # remap and >180 means that HMI_PAST is the previous CAR_ROT and HMI_FUTR is the current CAR_ROT
    # remap and <180 means that HMI_FUTR is the current CAR_ROT and HMI_PAST is the previous CAR_ROT
    """
    if remap and crln_obs > 180:
        car_rot = car_rot - 1
    """
    if remap and crln_obs > 180:
        # t defined by when HMI sees it (future/past)
        t0 = carrington_rotation_time(car_rot-1) # past / current
        t1 = carrington_rotation_time(car_rot)   # future

    else:        
        t0 = carrington_rotation_time(car_rot)  # future / current
        t1 = carrington_rotation_time(car_rot+1) # future end point

    hmi_times = "%s-%s" %(t0.datetime.strftime("%Y.%m.%d_%H:%M:%S_TAI"), t1.datetime.strftime("%Y.%m.%d_%H:%M:%S_TAI"))
    hmi_data, n = get_drms_keywords(hmi_times, "hmi.m_720s") #"mps_loeschl.Ml_remap_720s")

    if verbose: print('Mapping PHI to CR %s in HMI period %s...' %(car_rot, hmi_times))

    for line in hmi_data:    
        trec_hmi = np.append(trec_hmi, datetime.strptime(line['T_REC'], "%Y.%m.%d_%H:%M:%S_TAI"))
        crln_hmi = np.append(crln_hmi, float(line['CRLN_OBS']))
        dt_hmi   = np.append(dt_hmi, ((trec_hmi[-1] - t0.datetime).days +(trec_hmi[-1] - t0.datetime).seconds/(3600*24)))    

    return crln_hmi, dt_hmi, t0, car_rot


def interp_phi2hmi(crln_obs, crln_hmi, dt_hmi, t0):
    # Interpolate T_REC of PHI CRLN_OBS onto HMI CRLN_OBS
    
    t_interp = timedelta(days=np.interp(crln_obs, crln_hmi, dt_hmi, period=360))
    trec_phi = t0.datetime+t_interp
    trec_hmi = trec_phi.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    
    return trec_hmi
    

In [367]:
path = "../output/data/phi/feb2021/"
dbpath = "/data/solo/phi/data/fmdb/l1/%s/%s"

tmp = 0
files = os.listdir(path)
fitsfiles = [file for file in files if file.endswith(".fits")]

remap = True 

for file in fitsfiles:
    l2 = fits.open(path+file)
  
    print("Processing %s ..." %file)
    
    # David's L2 files were processed with an old header version. 
    # Find respective UPDATED L1 files and copy the missing keywords
    # L1 parent FILENAME is outdated. Find source file with updated processing
    
    # solo_L1_phi-fdt-ilam_obsdateTobstime_processingdate_noclue.fits.gz
    # solo_L1_phi-fdt-ilam_20210205T200002_V202108301608C_0142050411.fits.gz
    
    # Extract outdated filename
    date = l2[0].header['FILENAME'][21:29] # eg '20210219'
    date_path = "%s-%s-%s" %(date[:4], date[4:6], date[6:8])

    # Find updated source file from observation date/time of outdated source file
    tmpfiles = os.listdir(dbpath %(date_path, ""))
    for tmpfile in tmpfiles:
        if l2[0].header['FILENAME'][:37] in tmpfile: # FILENAME[:37] eg: 'solo_L1_phi-fdt-ilam_20210205T200002_'
            break
    
    prim = fits.PrimaryHDU()
    l2drms = fits.CompImageHDU(data=l2[0].data.astype(np.int32))

    # open updated source file
    l1 = fits.open(dbpath %(date_path, tmpfile))

    l2drms.header.append(('', '', ''), end=True)
    l2drms.header.append(('', '  / HMI Compatibility', ''), end=True)
    
    # TODO TEMPORARY: Fix CAR_ROT bug
    if l2[0].header['CAR_ROT'] == 2239: 
        l2[0].header['CAR_ROT'] = 2240
        
    # T_REC / T_OBS
    #DATE-AVG= '2021-02-05T20:00:45.616' / [UTC] Average time of observation     
    #T_OBS   = '2021.02.28_07:11:55.437_TAI' / [TAI] nominal time 
    #T_REC   = '2021.02.28_07:12:00.000_TAI' / [TAI] Slot time    
    # Conversion for date format / UTC to TAI / round to the next 12 minute slot for T_REC
    trec = datetime.strptime(l2[0].header['DATE-AVG'],   "%Y-%m-%dT%H:%M:%S.%f")
    tobs = datetime.strptime(l2[0].header['DATE-AVG'],   "%Y-%m-%dT%H:%M:%S.%f")
    
    #utc2tai = timedelta(0, 37)                 # use for utc2tai conversion
    #trec = trec + utc2tai                      # use for utc2tai conversion
    trec = trec.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    tobs = tobs.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    
    # T_REC Interpolation    
    crln_obs = float(l2[0].header['CRLN_OBS'])
    car_rot  = int(l2[0].header['CAR_ROT'])
    
    if car_rot != tmp:
        # only do this loop at the beginning of each CAR_ROT
        crln_hmi, dt_hmi, t0, car_rot = prep_hmi_interp(crln_obs, car_rot, remap=remap)
        trec_hmi = interp_phi2hmi(crln_obs, crln_hmi, dt_hmi, t0)
        
        #TODO
        # change prep_hmi_interp() to return t0 and t1 with a logic that t0 is always the previous HMI reference
        #hmi_past = interp_phi2hmi(crln_obs, crln_hmi, dt_hmi, t0)
        #hmi_futr = interp_phi2hmi(crln_obs, crln_hmi, dt_hmi, t1)
        # save both keywords for orientation
        # use as T_REC according to closest CAR_ROT
        
        tmp = car_rot
    else:
        trec_hmi = interp_phi2hmi(crln_obs, crln_hmi, dt_hmi, t0)

    l2drms.header.append(('T_REC', trec, ''), end=True)
    l2drms.header.append(('T_OBS', tobs, ''), end=True)  # required for JV2TS
    
    l2drms.header.append(('T_REC_EPOCH', '1993.01.01_00:00:00_TAI', 'Time of origin'), end=True)
    l2drms.header.append(('T_REC_STEP', 720.0,  'ts_eq step'), end=True)
    
    # these two keywords are redundant with T_REC and T_OBS
    l2drms.header.append(('TREC_HMI', trec_hmi, ''), end=True) # HMI interpolated T_REC
    l2drms.header.append(('TREC_PHI', tobs, ''),     end=True) # real PHI observation date as backup
    
    # DATE
    l2drms.header.append(('DATE', l2[0].header['DATE'], "Date and time of FITS file creation, in UTC, in ISO-8601 format 'yyyy-mm-ddThh:mm:ss.sss'"), end=True)
    
    # DATE-OBS
    # DATE-BEG= '2021-02-05T20:00:02.906' / [UTC] Start time of observation 
    # DATE-OBS= '2021-02-28T07:10:33.400' / [ISO] Observation date {DATE__OBS}   
    # Conversion from BEG to OBS. Conversion from UTC to ISO
    date_obs = datetime.strptime(l2[0].header['DATE-BEG'],   "%Y-%m-%dT%H:%M:%S.%f")
    date_obs = date_obs.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    l2drms.header.append(('DATE-OBS', date_obs, 'DATE-OBS = DATE-AVG - EXPTIME/2.0'), end=True)

    # CADENCE
    # Missing, directly copy from HMI as a dummy value or extract daily? cadence from filenames
    l2drms.header.append(('CADENCE', 720.0, '[seconds] Observation cadence - DUMMY VALUE'), end=True)
    
    # TELESCOP
    l2drms.header.append(('TELESCOP', l2[0].header['TELESCOP'], 'For PHI: SOLO/PHI/FDT or SOLO/PHI/HRT - For HMI: SDO/HMI'), end=True)
    
    # INSTRUME
    l2drms.header.append(('INSTRUME', l2[0].header['INSTRUME'], 'For PHI: PHI - For HMI: HMI_SIDE1, HMI_FRONT2, or HMI_COMBINED'), end=True)
    
    # WAVELNTH
    l2drms.header.append(('WAVELNTH', 6173.341, 'For PHI/HMI: 6173.3 Angstroms'), end=True)
    
    # QUALITY
    # Missing. add as dummy value = 0. Quality index of data used to genrate fd.B_720s (0x00000000) is good quality, (any nonzero value) should be investiaged in data documentation
    l2drms.header.append(('QUALITY', 0, 'Level 1.5 Quality - DUMMY VALUE'), end=True)

    # BUNIT
    l2drms.header.append(('BUNIT', l2[0].header['BUNIT'], 'BUNIT: physical units of each data segment'), end=True)
    
    # HISTORY
    # TODO FIX FILENAME ONCE WE HAVE GOOD L2 HEADERS
    l2drms.header.append(('HISTORY', 'DRMS compatible FITS created from %s' %l1[0].header['FILENAME'], 'History of data'), end=True)
    
    # COMMENT
    #l2drms.header.append(('COMMENT', l2[0].header['COMMENT'], 'Commentary on the data'), end=True)
    
    # CTYPE1
    l2drms.header.append(('CTYPE1', l2[0].header['CTYPE1'], 'CTYPE1: HPLN-TAN (SOLARX)'), end=True)
    
    # CTYPE2
    l2drms.header.append(('CTYPE2', l2[0].header['CTYPE2'], 'CTYPE2: HPLN-TAN (SOLARY)'), end=True)
    
    #CRPIX1
    l2drms.header.append(('CRPIX1', l2[0].header['CRPIX1'], 'CRPIX1: location of the Sun center in CCD x direction'), end=True)
    
    #CRPIX2
    l2drms.header.append(('CRPIX2', l2[0].header['CRPIX2'], 'CRPIX2: location of the Sun center in CCD y direction'), end=True)
    
    #CRVAL1
    l2drms.header.append(('CRVAL1', l2[0].header['CRVAL1'], 'CRVAL1: x origin - center of the solar disk'), end=True)
    
    #CRVAL2
    l2drms.header.append(('CRVAL2', l2[0].header['CRVAL2'], 'CRVAL2: y origin - center of the solar disk'), end=True)
    
    #CDELT1
    l2drms.header.append(('CDELT1', l2[0].header['CDELT1'], 'Image scale in the x direction'), end=True)
    
    #CDELT2
    l2drms.header.append(('CDELT2', l2[0].header['CDELT2'], 'Image scale in the y direction'), end=True)
    
    #CUNIT1
    l2drms.header.append(('CUNIT1', l2[0].header['CUNIT1'], 'CUNIT1: arcsec'), end=True)
    
    #CUNIT2
    l2drms.header.append(('CUNIT2', l2[0].header['CUNIT2'], 'CUNIT2: arcsec'), end=True)
    
    # CROTA 
    # CROTA2
    # CROTA was renamed to CROTA2 in Dietmar's current L2 header. Rename here for now. CHECK IF THE ROTATION MAKES SENSE AFTER THE PROJECTION by comparing the projections of equal CRLN_OBS in PHI and HMI
    l2drms.header.append(('CROTA2', l2[0].header['CROTA'], '[deg] Rotation angle'), end=True)

    #CRDER1
    #l2drms.header.append(('CRDER1', l1[0].header['CRDER1'], 'CRDER1: estimate of random error in coordinate x'), end=True)
    
    #CRDER2
    #l2drms.header.append(('CRDER2', l1[0].header['CRDER2'], 'CRDER2: estimate of random error in coordinate y'), end=True)
    
    #CSYSER1
    #l2drms.header.append(('CSYSER1', l2[0].header['CSYSER1'], 'CSYSER1: estimate of systematic error in coordinate x'), end=True)
    
    #CSYSER2
    #l2drms.header.append(('CSYSER2', l2[0].header['CSYSER2'], 'CSYSER2: estimate of systematic error in coordinate y'), end=True)
    
    #WCSNAME
    l2drms.header.append(('WCSNAME', l2[0].header['WCSNAME'], 'WCS system name'), end=True)
    
    #DSUN_OBS
    l2drms.header.append(('DSUN_OBS', l2[0].header['DSUN_OBS'], 'Distance from SDO to Sun center.'), end=True)
    
    #RSUN_REF
    l2drms.header.append(('RSUN_REF', l2[0].header['RSUN_REF'], 'Reference radius of the Sun: 696,000,000.0 m'), end=True)
    
    #CRLN_OBS
    l2drms.header.append(('CRLN_OBS', l2[0].header['CRLN_OBS'], 'Carrington longitude of HMI'), end=True)
    
    #CRLT_OBS
    l2drms.header.append(('CRLT_OBS', l2[0].header['CRLT_OBS'], 'Carrington latitude of HMI'), end=True)
    
    #CAR_ROT        
    l2drms.header.append(('CAR_ROT', car_rot, 'Carrington rotation number of CRLN_OBS'), end=True)
    
    # OBS_VW
    # OBS_VR
    # OBS_VN
    # Both are missing in the OLD HEADER (David's) but should't in the future. Fetch keywords from L1 data, which is the one listed in the processed FILENAME history keyword
    # eg.: FILENAME solo_L1_phi-fdt-ilam_20210205T200002_V202108301608C_0142050411.fits
    l2drms.header.append(('OBS_VR', l1[0].header['OBS_VR'], '[m/s] Radial velocity of S/C relative to Sun   '), end=True)
    l2drms.header.append(('OBS_VW', l1[0].header['OBS_VW'], '[m/s] Westward velocity of S/C relative to Sun '), end=True)
    l2drms.header.append(('OBS_VN', l1[0].header['OBS_VN'], '[m/s] Northward velocity of S/C relative to Sun'), end=True)

    # RSUN_ARC
    # RSUN_OBS
    # Same keyword. Save RSUN_ARC as RSUN_OBS for HMI
    l2drms.header.append(('RSUN_OBS', l2[0].header['RSUN_ARC'], '[arcsec] angular radius of Sun.'), end=True)

    # DATAVALS
    # Actual number of data values in images (pixels)
    l2drms.header.append(('DATAVALS', l2[0].header['NAXIS1']*l2[0].header['NAXIS2'], 'Actual number of data values in images'), end=True)

    # MISSVALS
    # Missing values: TOTVALS - DATAVALS
    l2drms.header.append(('MISSVALS', 0, 'Missing values: TOTVALS - DATAVALS'), end=True)
    
    # DATAMIN
    l2drms.header.append(('DATAMIN', l2[0].header['DATAMIN'], 'Minimum value from pixels within 99% of solar radius'), end=True)
    
    # DATAMAX
    l2drms.header.append(('DATAMAX', l2[0].header['DATAMAX'], 'Maximum value from pixels within 99% of solar radius'), end=True)
    
   
    hdul = fits.HDUList([prim, l2drms])
    #hdul.writeto(path+'drms/%s_drms.fits' %file[:-5], overwrite=True)
    l1.close()
    break
print('Done')

#ef make_fits(phi, hmi, dtype):
#   # phi ... image
#   # hmi ... image

#   prim = fits.PrimaryHDU(data=hmi[0].data, header=hmi[0].header)
#   cimg = fits.CompImageHDU(data=phi.astype(dtype), header=hmi[1].header[:-2]) # -2 skips old checksum and datasum
#   hdul = fits.HDUList([prim, cimg])
#       
#   return hdul

Processing solo_L2_phi-fdt-blos_20210205T200002_V202107130921C_0142050411.fits ...
True 349.87855
in here
Done


In [368]:
l2drms.header

XTENSION= 'IMAGE   '           / Image extension                                
BITPIX  =                   32 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 1280                                                  
NAXIS2  =                 1280                                                  
PCOUNT  =                    0 / number of parameters                           
GCOUNT  =                    1 / number of groups                               
                                                                                
          / HMI Compatibility                                                   
T_REC   = '2021.02.05_20:00:45_TAI'                                             
T_OBS   = '2021.02.05_20:00:45_TAI'                                             
HIERARCH T_REC_EPOCH = '1993.01.01_00:00:00_TAI' / Time of origin               
HIERARCH T_REC_STEP = 720.0 

In [318]:
phi_data # new

[{'T_REC': '2021.02.05_20:00:45_TAI',
  'CRLN_OBS': '349.878540',
  'CAR_ROT': 2241,
  'TREC_HMI': '2021.02.19_09:10:57_TAI',
  'TREC_PHI': '2021.02.05_20:00:45_TAI'},
 {'T_REC': '2021.02.09_12:30:45_TAI',
  'CRLN_OBS': '309.480194',
  'CAR_ROT': 2241,
  'TREC_HMI': '2021.02.22_10:48:39_TAI',
  'TREC_PHI': '2021.02.09_12:30:45_TAI'},
 {'T_REC': '2021.02.10_12:30:45_TAI',
  'CRLN_OBS': '298.545929',
  'CAR_ROT': 2241,
  'TREC_HMI': '2021.02.23_06:42:52_TAI',
  'TREC_PHI': '2021.02.10_12:30:45_TAI'},
 {'T_REC': '2021.02.11_12:30:45_TAI',
  'CRLN_OBS': '287.611237',
  'CAR_ROT': 2241,
  'TREC_HMI': '2021.02.24_02:37:27_TAI',
  'TREC_PHI': '2021.02.11_12:30:45_TAI'},
 {'T_REC': '2021.02.12_12:30:45_TAI',
  'CRLN_OBS': '276.671143',
  'CAR_ROT': 2241,
  'TREC_HMI': '2021.02.24_22:33:55_TAI',
  'TREC_PHI': '2021.02.12_12:30:45_TAI'},
 {'T_REC': '2021.02.13_12:30:45_TAI',
  'CRLN_OBS': '265.720734',
  'CAR_ROT': 2241,
  'TREC_HMI': '2021.02.25_18:32:25_TAI',
  'TREC_PHI': '2021.02.13_12:30:45

In [237]:
phi_data # old

[{'T_REC': '2021.02.05_20:00:45_TAI',
  'CRLN_OBS': '349.878540',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.19_09:10:54_TAI'},
 {'T_REC': '2021.02.09_12:30:45_TAI',
  'CRLN_OBS': '309.480194',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.22_10:48:39_TAI'},
 {'T_REC': '2021.02.10_12:30:45_TAI',
  'CRLN_OBS': '298.545929',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.23_06:42:54_TAI'},
 {'T_REC': '2021.02.11_12:30:45_TAI',
  'CRLN_OBS': '287.611237',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.24_02:37:27_TAI'},
 {'T_REC': '2021.02.12_12:30:45_TAI',
  'CRLN_OBS': '276.671143',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.24_22:33:55_TAI'},
 {'T_REC': '2021.02.13_12:30:45_TAI',
  'CRLN_OBS': '265.720734',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.25_18:32:23_TAI'},
 {'T_REC': '2021.02.14_12:30:45_TAI',
  'CRLN_OBS': '254.755127',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.26_14:32:06_TAI'},
 {'T_REC': '2021.02.16_12:30:45_TAI',
  'CRLN_OBS': '232.760178',
  'CAR_ROT': '2241',
  '

In [223]:
phi_data

[{'T_REC': '2021.02.05_20:00:45_TAI',
  'CRLN_OBS': '349.878540',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.19_09:10:54_TAI'},
 {'T_REC': '2021.02.09_12:30:45_TAI',
  'CRLN_OBS': '309.480194',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.22_10:48:39_TAI'},
 {'T_REC': '2021.02.10_12:30:45_TAI',
  'CRLN_OBS': '298.545929',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.23_06:42:54_TAI'},
 {'T_REC': '2021.02.11_12:30:45_TAI',
  'CRLN_OBS': '287.611237',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.24_02:37:27_TAI'},
 {'T_REC': '2021.02.12_12:30:45_TAI',
  'CRLN_OBS': '276.671143',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.24_22:33:55_TAI'},
 {'T_REC': '2021.02.13_12:30:45_TAI',
  'CRLN_OBS': '265.720734',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.25_18:32:23_TAI'},
 {'T_REC': '2021.02.14_12:30:45_TAI',
  'CRLN_OBS': '254.755127',
  'CAR_ROT': '2241',
  'T_REC_HMI': '2021.02.26_14:32:06_TAI'},
 {'T_REC': '2021.02.16_12:30:45_TAI',
  'CRLN_OBS': '232.760178',
  'CAR_ROT': '2241',
  '

In [218]:
t_interp

datetime.timedelta(days=9, seconds=57130, microseconds=665983)

In [219]:
t0.datetime+t_interp

datetime.datetime(2021, 2, 28, 6, 35, 7, 708972)

In [200]:
(trec_hmi[-1] - t0.datetime)

datetime.timedelta(days=9, seconds=59342, microseconds=957011)

In [205]:
((trec_hmi[-1] - t0.datetime).days +(trec_hmi[-1] - t0.datetime).seconds/(3600*24))

9.686828703703704

In [209]:
phi_data

[{'T_REC': '2021.02.05_20:00:45_TAI',
  'CRLN_OBS': '349.878540',
  'CAR_ROT': '2241',
  'T_REC_HMI': 0.7694139806600262},
 {'T_REC': '2021.02.09_12:30:45_TAI',
  'CRLN_OBS': '309.480194',
  'CAR_ROT': '2241',
  'T_REC_HMI': 3.837298581125085},
 {'T_REC': '2021.02.10_12:30:45_TAI',
  'CRLN_OBS': '298.545929',
  'CAR_ROT': '2241',
  'T_REC_HMI': 4.666639056170476},
 {'T_REC': '2021.02.11_12:30:45_TAI',
  'CRLN_OBS': '287.611237',
  'CAR_ROT': '2241',
  'T_REC_HMI': 5.496181791551054},
 {'T_REC': '2021.02.12_12:30:45_TAI',
  'CRLN_OBS': '276.671143',
  'CAR_ROT': '2241',
  'T_REC_HMI': 6.3270606356921615},
 {'T_REC': '2021.02.13_12:30:45_TAI',
  'CRLN_OBS': '265.720734',
  'CAR_ROT': '2241',
  'T_REC_HMI': 7.15933523815369},
 {'T_REC': '2021.02.14_12:30:45_TAI',
  'CRLN_OBS': '254.755127',
  'CAR_ROT': '2241',
  'T_REC_HMI': 7.9924735794663455},
 {'T_REC': '2021.02.16_12:30:45_TAI',
  'CRLN_OBS': '232.760178',
  'CAR_ROT': '2241',
  'T_REC_HMI': 9.661234559984091}]

In [122]:
np.interp(phi_data[0]['CRLN_OBS'], crln_hmi, dt_hmi, period=360)

0.7689968164298508

In [151]:
test = Time(datetime.strptime(line['T_REC'], "%Y.%m.%d_%H:%M:%S_TAI"), format='datetime', scale='tai').to_datetime

In [152]:
test

<bound method Time.to_datetime of <Time object: scale='tai' format='datetime' value=2021-02-28 07:12:00>>

In [126]:
phi_data[0]['T_REC_HMI'] = np.interp(phi_data[0]['CRLN_OBS'], crln_hmi, dt_hmi, period=360)

In [141]:
phi_data[0]['T_REC_HMI']

<Time object: scale='tai' format='jd' value=0.7689968164298508>

In [148]:
trec_hmi[-1] - t0

<TimeDelta object: scale='tai' format='jd' value=9.686411539473529>

In [124]:
phi_dat{"HMI_T_REC:" []}

SyntaxError: invalid syntax (<ipython-input-124-4c7cd60879d4>, line 1)

In [108]:
dt_hmi.value

AttributeError: 'numpy.ndarray' object has no attribute 'value'

In [87]:
trec = datetime.strptime(trec_hmi[0], "%Y.%m.%d_%H:%M:%S_TAI")

In [90]:
trec = Time(datetime.strptime(trec_hmi, "%Y.%m.%d_%H:%M:%S_TAI"), format='datetime', scale='tai')

In [91]:
trec

<Time object: scale='tai' format='datetime' value=2021-02-19 09:00:00>

In [132]:
Time(phi_data[0]['T_REC_HMI'], format='jd', scale='tai')

<Time object: scale='tai' format='jd' value=0.7689968164298508>

In [75]:
TimeDelta(trec_hmi[0], t0)

ValueError: Input values did not match the format class jd:
TypeError: for jd class, input should be doubles, string, or Decimal, and second values are only allowed for doubles.

In [76]:
TimeDelta

astropy.time.core.TimeDelta

In [65]:
t00 = carrington_rotation_time(int(phi_data[i]['CAR_ROT'])-1)

In [66]:
t00

<Time object: scale='utc' format='iso' value=2021-01-22 06:31:17.266>

In [57]:
trec_hmi

array(['2021.02.19_09:00:00_TAI', '2021.02.22_10:48:00_TAI',
       '2021.02.23_06:48:00_TAI', '2021.02.24_02:36:00_TAI',
       '2021.02.24_22:36:00_TAI', '2021.02.25_18:36:00_TAI',
       '2021.02.26_14:36:00_TAI', '2021.02.28_07:12:00_TAI'], dtype='<U32')

In [58]:
crln_hmi

array(['349.978424', '309.486389', '298.499481', '287.624664',
       '276.652252', '265.687927', '254.719742', '232.422836'],
      dtype='<U32')